# 三条鱼 3D 关键点：姿态 PCA、微状态、转移矩阵与 Infomap 聚类（K / P / KP）

本 notebook 对 1 雌 2 雄（F、M1、M2）的 3 节点骨架（head = 嘴、pec = 胸鳍点、tail = 尾）做：

Step 0 读数据与连续片段 → Step 1 PCA → Step 2 画出主成分 → Step 3 ACF → Step 4 离散化 → Step 5 转移矩阵 → Step 5b Infomap → Step 6 K 与 P 一致性 → Step 7 手动标记验证（可选）→ Step 8 总览。

- 所有参数在第一个代码单元。第二个代码单元是公共函数（不是分析步骤）。
- 每一步的中间结果写到 `data/`，后面的步骤从 `data/` 读。改了某一步之后的参数，只需重跑 setup 单元和那一步及其之后的步骤。
- 只有 Step 7 读取手动标记；它之前的所有步骤（包括聚类、τ、分格、PC 的选择）都不看标记。
- notebook 不下"一致/不一致/聚类对不对"的结论，只写数值和图。

## 输出目录（扁平）

```
<runs_dir>/behavior_features/outputs/<RUN_NAME>/
  index.html  summary.csv  figures_index.csv  params.json
  figures/  thumbs/  compare/  data/  validation/
```
命名：`<步骤号>_<步骤名>__<单元>__<参数>__<分析集>.<扩展名>`，例如 `05_transition_joint__FM1__tau7-bins10-pcs1_2_3__K.png`。参数用 `key数值`，以 `-` 连接；没有参数写 `base`。

## 假设（不明确的地方，我的选择；请核对）

1. **数据**：分析集 K = `20251225_K_1_1_FCMC`、`20251225_K_1_2_FCMC`；P = `20260112_P_1_1_FCMC`、`20260113_P_1_2_FCMC`（与上一版 `notebooks/pose_pca_kp.ipynb` 相同；第 3 天是留出的测试视频，不读）。KP = 四个视频合并，单独拟合 PCA、单独分格、单独聚类。
2. **坐标与身份**：review package 的 `nodes3d`，单位毫米，z 竖直向上。F/M1/M2 来自身份审核账本。审核未接受的帧（包括每个视频开头分析起点之前的那一段）在读入时已经是 NaN。
3. **连续片段**：该单元所需的坐标全部有限的连续帧；视频边界一定断开。版本 A 另外要求两条鱼 pec 的水平距离 ≥ `MIN_XY_MM`（否则 X' 没有定义）；这些帧在缺口统计里单独算作 "pec XY too close"。
4. **缺口的两种来源可以区分**："手动删除" = 审核/身份账本不接受的帧；"追踪丢失" = 被接受但坐标是 NaN 的帧。缺口统计包含视频开头分析起点之前的那一段。
5. **可选预处理的顺序**（默认都关）：骨架阈值 → 线性插值（不跨手动删除的帧）→ Savitzky-Golay 平滑（每个有限片段内做，端点用 `mode='interp'`，短于窗口的片段不平滑）→ 体长归一化。骨架距离直方图总是用原始毫米数据画，`HEAD_PEC_MAX`、`PEC_TAIL_MAX` 也是毫米。阈值剔除的是超阈值的那条鱼在那一帧的全部坐标。
6. **版本 A 的特征顺序**（固定）：鱼1 head x'y'z'，鱼1 tail x'y'z'，鱼2 head，鱼2 tail，鱼1 pec x'、z'。X' 从鱼1 pec 指向鱼2 pec，所以鱼1 pec 的 x' 总是负的。
7. **版本 B**：默认是 27 个原始坐标（F、M1、M2 × head、pec、tail × x、y、z），图题写 "raw coordinates"。`TRIPLE_COROTATE = True` 时用共转坐标系：原点 = 三个 pec 的平均；Z' = Z；X' = F 的 pec 指向两条雄鱼 pec 中点的水平方向；Y' 补全右手系。这个定义是本 notebook 自己选的，论文里没有。打开后 `F_pec_y'` 恒为 0（方差为 0 的特征）。
8. **PCA**：协方差矩阵特征分解，只减均值（`PCA_STANDARDIZE=False`）。`FIX_PC_SIGN=True`：每个 PC 里绝对值最大的载荷取正，得分同步翻转。
9. **Step 2 的相关**：Pearson。角度用度。朝向角是周期量，在 ±180° 处断开，会削弱线性相关。
10. **ACF**：每个 PC 得分在整个分析集的有效帧上标准化（均值 0、方差 1）；每个片段内 c_s(l) = 片段内 n−l 对乘积的平均；按片段长度 n 加权平均（只用 n > l 的片段）；片段内不再减均值，所以 lag 0 恰好是 1。1/e 时间 = 第一次向下穿越的线性插值；没有穿越则留空。
11. **分格**：等宽；最大值落在最后一格。百分位模式下超出范围的帧不参与（状态 −1），并打印占比。joint 状态编号：第一个变量变化最快，state = b1 + N·b2 + N²·b3。
12. **θ 的符号**（未与论文核对）：从"自己 pec → 对方 pec"的水平方向，逆时针（从上往下看，z 向上）转到"自己 pec → 自己 head"的水平方向为正；0 = 头朝向对方；范围 [−π, π]。
13. **`D_MAX = 20` 和坐标同单位**：默认坐标是毫米，所以是 20 mm（约 0.8 体长），D_PP 超过它的帧会被排除，而且这样的帧可能占大多数。上一版用的是 200 mm。打印出的排除占比请核对；只在 `STATE_VARS="physical"` 时起作用（默认是 `"pcs"`）。
14. **`SYMMETRIZE_M1M2`**：θ 对调后的副本作为一条独立的时间序列加入（只在副本内部配对，不跨副本），占用计数因此加倍；簇编号只对原始帧输出。
15. **转移矩阵**：所有视频、所有时刻合并计数。检查里的"对角线占比" = 对角线计数 / 总计数。
16. **Infomap**：去掉行和列计数都为 0 的状态；边权 = 行归一化后的概率；`KEEP_SELF_LOOPS=False` 时先去掉对角线再归一化。簇按覆盖帧数降序重新编号（0 = 覆盖最多）。状态有效但没有进入图的帧（没有任何有效配对）簇号为 −1。python-igraph 的 `community_infomap` 没有种子参数；每次调用前设 `random.seed(INFOMAP_SEED)`（igraph 用 Python 的 random 模块）。
17. **簇×簇图**：状态级计数聚合到簇，再按行归一化；未被选中的簇合并成一行一列 "rest"。"去掉对角线"的版本把对角线遮成白色，其它数值不变，色标只按非对角线的数值取。状态级块对角图同理（遮掉状态的对角线）。
18. **打乱参照**：在进入图的状态之间随机置换簇标签（每个簇的状态数不变），重算簇内转移占比，重复 `N_SHUFFLE` 次。
19. **Step 6**：余弦相似度取绝对值；每个 K 的 PC 独立地在 P 里找最相似的 PC（可能两个 K 的 PC 对到同一个 P 的 PC）。跨组应用时用源组的 PCA 均值、载荷和分格边界；超出边界的帧不参与占用分布和转移计数。JS 距离 = `scipy.spatial.distance.jensenshannon`（base 2，范围 0–1）。跨组应用只用原始帧，不加对称副本。按天拆分：组内的视频按天排序，前一半天 vs 后一半天（这里每组正好两天）。同一组内不同天是同三条鱼，不是独立样本。
20. **Step 7**：`VALIDATION_CLUSTERS="dev"`（默认）：用开发组自己的 PCA、分格和簇（状态→簇），测试组投影到开发组的状态空间再查簇；`"KP"`：用 KP 的聚类（聚类时看到了测试组的数据，但没有看标记）。一帧同时有两个类别的标记时丢弃这一帧并计数。对每个 τ ∈ `TAUS` 各做一次。
21. **命名的补充**：和单元无关的输出（骨架距离直方图）单元写 `all`，分析集写 `all`；Step 6 的分析集写 `KvsP`；Step 7 写 `K-to-P` / `P-to-K`；Step 5b 的步骤号写 `05b`。只对三条鱼起作用的选项（`corot`）在 by_unit 对比图里不阻止配对。

In [1]:
# ============================== Parameters ==============================
# Every analysis setting lives in this cell. Optional switches default to off unless the
# specification says otherwise. Units of lengths follow the coordinates (mm, or body
# lengths when NORM_BL is on) unless the name ends in _MM.
from pathlib import Path

RUN_NAME = "20261107_pose_states"
OUTPUT_ROOT = None          # None -> <runs_dir>/behavior_features/outputs (paths config)
RUN_SETS = ["K", "P", "KP"]
RUN_UNITS = ["FM1", "FM2", "M1M2", "triple"]   # "M2M1" is also available
K_EXPERIMENTS = ["20251225_K_1_1_FCMC", "20251225_K_1_2_FCMC"]
P_EXPERIMENTS = ["20260112_P_1_1_FCMC", "20260113_P_1_2_FCMC"]
DPI = 200
DPI_BIG = 300
THUMB_WIDTH = 400

# --- Step 0: segments, gaps, optional preprocessing
INTERPOLATE = False          # linear fill of short gaps (never across manually removed frames)
INTERP_MAX_FRAMES = 8
SMOOTH_SG = False            # Savitzky-Golay inside each finite run
SG_WINDOW = 7
SG_ORDER = 2
NORM_BL = False              # divide by body length (per video, per fish); adds "norm_bl" to labels
HEAD_PEC_MAX = None          # mm; fill in after looking at the skeleton-distance histogram
PEC_TAIL_MAX = None          # mm
MIN_XY_MM = 1.0              # version A: horizontal pec-pec distance below this has no X' direction
TRIPLE_COROTATE = False      # version B: co-rotating frame (see assumption 7); False = raw coordinates

# --- Step 1: PCA
PCA_STANDARDIZE = False
FIX_PC_SIGN = True
PC_GROUPS = {}               # fill by hand after the loading heatmap, e.g. {"distance": [1, 2], "shape": [3, 4]}
LOAD_THR = 0.3
CUMVAR_YMIN = 50.0

# --- Step 2: draw the PCs
N_SHOW = 3
K_SIGMAS = (-2, 0, 2)

# --- Step 3: ACF
MAX_LAG_S = 5.0
N_ACF = 10

# --- Step 4: discretisation
STATE_VARS = "pcs"           # "pcs" or "physical" (physical: version A only)
PCS_FOR_STATE = [1, 2, 3]
N_BINS = 10
BIN_BY_PERCENTILE = False
PERCENTILE = (0.5, 99.5)
D_MAX = 20.0                 # same unit as the coordinates (see assumption 13)
N_BINS_D = 10
N_BINS_THETA = 10
SYMMETRIZE_M1M2 = False
DO_JOINT = True
DO_PER_PC = True

# --- Step 5: transition matrices
TAUS = [7, 17]               # frames

# --- Step 5b: Infomap
KEEP_SELF_LOOPS = True
N_TRIALS = 500
N_TRIALS_SCAN = 50
INFOMAP_SEED = 20261107
INFOMAP_WORKERS = 6          # parallel processes for independent Infomap runs
CLUSTER_SELECT = "top_coverage"   # or "all"
COVERAGE = 0.95
FILTER_RARE = True
MIN_VISITS = 50
TOP_ZOOM = 6
MAKE_HTML = False            # needs plotly
N_SHUFFLE = 100
TAU_SCAN = [1, 2, 3, 5, 7, 10, 12, 15, 17, 20, 25, 30, 35]

# --- Step 6: K vs P
N_CMP = 6
DO_DAY_REFERENCE = True

# --- Step 7: validation with manual labels (filled in by the user; step is skipped until then)
LABEL_SOURCE = "boris"       # "boris" | "events_csv" (video,target,label,start_s,end_s) | "frames_csv" (video,target,frame,label)
LABEL_PATH = None            # None -> repository BORIS search (archive/boris, boris_events_dir)
LABEL_GRANULARITY = "events" # "events" or "frame"
LABEL_TARGET = "pair"        # "fish" | "pair" | "scene"
LABEL_CLASSES = {}           # e.g. {"courtship": ["Courtship", "Approach"], "attack": ["Dash", "Chase"]}; rest -> "other"
LABEL_UNLABELED = "other"    # "other" or "ignore"
LABEL_FPS = None             # None -> same frame clock as the tracking (166 fps)
LABEL_PAIR_MAP = {}          # e.g. {"FM1": ["M1->F"], "FM2": ["M2->F"], "M1M2": ["M1->M2", "M2->M1", "M1<->M2"]}
VALIDATION_CLUSTERS = "dev"  # "dev" or "KP" (assumption 20)
DEV_SETS = ["K"]
TEST_SETS = ["P"]
MIN_PURITY = 0.5
TOL_S = 0.10
TOL_FRAC = 0.25
OVERLAP_THR = 0.5
DURATION_EDGES_S = [0.1, 0.75]
BOOT = 1000
SHIFT_N = 100
N_EXAMPLE = 2
VALIDATION_SEED = 20261107

PARAMETERS = {k: (str(v) if isinstance(v, Path) else v) for k, v in dict(globals()).items()
              if k.isupper() and k != "PARAMETERS"}
print(f"{len(PARAMETERS)} parameters; run {RUN_NAME}; sets {RUN_SETS}; units {RUN_UNITS}")

75 parameters; run 20261107_pose_states; sets ['K', 'P', 'KP']; units ['FM1', 'FM2', 'M1M2', 'triple']


## Setup（公共函数，不是分析步骤）

找到仓库根目录，读取路径配置，建立输出目录，定义 `save_fig` 和各步骤共用的函数。`params.json` 在这里写入参数、日期和库版本；Step 0 补上帧数，Step 5b 补上 Infomap 后端。重跑任何一步之前先运行这个单元。

In [2]:
import os
import sys
import json
import time
import platform
import importlib

REPO = Path.cwd().resolve()
while not (REPO / "pyproject.toml").exists():
    if REPO.parent == REPO:
        raise RuntimeError("repository root (pyproject.toml) not found above the working directory")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy import sparse

from scripts.analysis import microstate as M
from scripts.analysis import pose_states as S
from scripts.analysis.behavior_dataset import resolve_roots

importlib.reload(S)

_CONFIG, REVIEW_ROOT, _BF_ROOT = resolve_roots(None)
OUT_ROOT = Path(OUTPUT_ROOT) if OUTPUT_ROOT else Path(_BF_ROOT) / "outputs"
L = S.RunLayout(OUT_ROOT / RUN_NAME, dpi=DPI, thumb_width=THUMB_WIDTH)


def save_fig(fig, step, name, unit, params, set_name, dpi=None, folder="figures"):
    # The only way figures are written: automatic name, png, thumbnail, row in figures_index.csv.
    return L.save_fig(fig, step, name, unit, params, set_name, dpi=dpi, folder=folder)


SET_GROUPS = {"K": ["K"], "P": ["P"], "KP": ["K", "P"]}
GROUP_EXPS = {"K": list(K_EXPERIMENTS), "P": list(P_EXPERIMENTS)}
FISH_COLOR = {"F": "#c0392b", "M1": "#1f5fa8", "M2": "#2e8b3e"}
PALETTE = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2", "#B279A2", "#FF9DA6", "#9D755D"]
for _s in RUN_SETS:
    if _s not in SET_GROUPS:
        raise ValueError(f"unknown analysis set {_s}")
for _u in RUN_UNITS:
    if _u not in S.PAIR_UNITS and _u != "triple":
        raise ValueError(f"unknown unit {_u}")


def lib_versions():
    import matplotlib as _mpl, scipy as _sp, PIL as _pil
    out = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
           "scipy": _sp.__version__, "matplotlib": _mpl.__version__, "pillow": _pil.__version__}
    for mod in ("igraph", "infomap", "sklearn", "plotly"):
        try:
            out[mod] = getattr(importlib.import_module(mod), "__version__", "unknown")
        except ImportError:
            out[mod] = None
    return out


L.update_params("parameters", PARAMETERS)
L.update_params("run", {"run_name": RUN_NAME, "date": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "output_dir": str(L.root), "review_root": str(REVIEW_ROOT),
                        "paths_config": str(_CONFIG), "platform": platform.platform(),
                        "libraries": lib_versions()})


# ---------------------------------------------------------------- parameter labels
def pre_params():
    # Step 0 options: they change the data of every later step.
    return {"interp": INTERP_MAX_FRAMES if INTERPOLATE else None,
            "sg": f"{SG_WINDOW}o{SG_ORDER}" if SMOOTH_SG else None,
            "hpmax": HEAD_PEC_MAX, "ptmax": PEC_TAIL_MAX, "norm_bl": bool(NORM_BL)}


def seg_params(unit):
    p = dict(pre_params())
    if unit == "triple":
        p["corot"] = bool(TRIPLE_COROTATE)
    return p


def unit_params(unit):
    p = seg_params(unit)
    p["std"] = bool(PCA_STANDARDIZE)
    return p


def uses_physical(unit):
    return STATE_VARS == "physical" and unit != "triple"


def state_params(unit):
    if uses_physical(unit):
        p = {"phys": True, "bd": N_BINS_D, "bth": N_BINS_THETA, "dmax": D_MAX,
             "sym": bool(SYMMETRIZE_M1M2 and unit == "M1M2")}
        p.update(seg_params(unit))
    else:
        p = {"bins": N_BINS, "pcs": list(PCS_FOR_STATE), "pctl": bool(BIN_BY_PERCENTILE)}
        p.update(unit_params(unit))
    return p


def tau_params(unit, tau):
    return {"tau": int(tau), **state_params(unit)}


if STATE_VARS not in ("pcs", "physical"):
    raise ValueError("STATE_VARS must be 'pcs' or 'physical'")


# ---------------------------------------------------------------- units
def is_pair(unit):
    return unit in S.PAIR_UNITS


def unit_fish(unit):
    return S.PAIR_UNITS[unit] if is_pair(unit) else S.FISH


def coord_unit():
    return "BL" if NORM_BL else "mm"


def unit_title(unit):
    if is_pair(unit):
        a, b = S.PAIR_UNITS[unit]
        return f"{unit} ({a}->{b})"
    return "triple (co-rotating frame)" if TRIPLE_COROTATE else "triple (raw coordinates)"


def prime(unit):
    return "'" if (is_pair(unit) or TRIPLE_COROTATE) else ""


def required_finite(nodes, unit):
    if is_pair(unit):
        return np.isfinite(S.pair_points(nodes, unit)).all(axis=(1, 2))
    return np.isfinite(nodes).all(axis=(1, 2, 3))


def unit_pose(nodes, unit, min_xy):
    # Pair: (T, 6, 3) in the co-rotating pair frame. Triple: (T, 9, 3), raw or co-rotating.
    if is_pair(unit):
        return S.align_pair(S.pair_points(nodes, unit), min_xy)
    if TRIPLE_COROTATE:
        pose, valid = S.align_triple(nodes, min_xy)
        return pose.reshape(len(nodes), 9, 3), valid
    valid = np.isfinite(nodes).all(axis=(1, 2, 3))
    pose = nodes.reshape(len(nodes), 9, 3).astype(np.float64, copy=True)
    pose[~valid] = np.nan
    return pose, valid


def unit_features(nodes, unit, use):
    # Features of every frame; NaN outside `use` (the Step 0 segments).
    pose, _valid = unit_pose(nodes, unit, 1e-9)
    feat = S.pair_features(pose) if is_pair(unit) else pose.reshape(len(pose), 27)
    feat = np.array(feat, dtype=np.float64, copy=True)
    feat[~use] = np.nan
    pose = np.array(pose, dtype=np.float64, copy=True)
    pose[~use] = np.nan
    return feat, pose


def feature_names(unit):
    return S.pair_feature_names(unit) if is_pair(unit) else S.triple_feature_names(TRIPLE_COROTATE)


def physical_vars(nodes, unit):
    # D_PP = 3D pec-pec distance; theta = signed horizontal angle from (own pec -> other pec)
    # to (own pec -> own head), counter-clockwise positive seen from above, 0 = facing the other fish.
    a, b = S.PAIR_UNITS[unit]
    ia, ib = S.FISH.index(a), S.FISH.index(b)
    pa, pb = nodes[:, ia, S.PEC], nodes[:, ib, S.PEC]
    ha, hb = nodes[:, ia, S.HEAD], nodes[:, ib, S.HEAD]

    def theta(head, pec, other):
        h = head[:, :2] - pec[:, :2]
        r = other[:, :2] - pec[:, :2]
        ang = np.arctan2(r[:, 0] * h[:, 1] - r[:, 1] * h[:, 0], np.sum(r * h, axis=1))
        bad = (np.linalg.norm(h, axis=1) == 0) | (np.linalg.norm(r, axis=1) == 0)
        ang[bad] = np.nan
        return ang

    return {"D_PP": np.linalg.norm(pb - pa, axis=1), f"theta_{a}": theta(ha, pa, pb), f"theta_{b}": theta(hb, pb, pa)}


def bend_deg(nodes, i):
    return np.degrees(M.bend_angle(nodes[:, i, S.HEAD], nodes[:, i, S.PEC], nodes[:, i, S.TAIL]))


# ---------------------------------------------------------------- loading data written by earlier steps
_CACHE = {}


def load_group(group):
    key = ("group", group, S.param_label(pre_params()))
    if key not in _CACHE:
        _CACHE[key] = L.load_npz("00", "nodes", "all", pre_params(), group)
    return _CACHE[key]


def load_set(set_name):
    key = ("set", set_name, S.param_label(pre_params()))
    if key in _CACHE:
        return _CACHE[key]
    parts = [load_group(g) for g in SET_GROUPS[set_name]]
    offsets = np.cumsum([0] + [len(p["names"]) for p in parts])
    out = {
        "nodes": np.concatenate([p["nodes"] for p in parts]).astype(np.float64),
        "video": np.concatenate([p["video"].astype(np.int64) + o for p, o in zip(parts, offsets[:-1])]),
        "frame": np.concatenate([p["frame"] for p in parts]).astype(np.int64),
        "blocked": np.concatenate([p["blocked"] for p in parts]),
        "names": np.concatenate([p["names"] for p in parts]).astype(str),
        "fps_v": np.concatenate([p["fps"] for p in parts]).astype(float),
        "day_v": np.concatenate([p["day"] for p in parts]).astype(int),
        "group_v": np.concatenate([[g] * len(p["names"]) for g, p in zip(SET_GROUPS[set_name], parts)]),
    }
    out["fps"] = out["fps_v"][out["video"]]
    _CACHE[key] = out
    return out


def set_fps(data):
    fps = np.unique(data["fps_v"])
    if len(fps) != 1:
        raise RuntimeError(f"videos have different frame rates: {fps}")
    return float(fps[0])


def load_seg(unit, set_name):
    segs, offset = [], 0
    for g in SET_GROUPS[set_name]:
        s = L.load_npz("00", "segments", unit, seg_params(unit), g)["seg"].astype(np.int64)
        s = np.where(s >= 0, s + offset, -1)
        if (s >= 0).any():
            offset = int(s.max()) + 1
        segs.append(s)
    return np.concatenate(segs)


def load_model(unit, set_name):
    return L.load_npz("01", "pca_model", unit, unit_params(unit), set_name)


def load_scores(unit, set_name):
    return L.load_npz("01", "pc_scores", unit, unit_params(unit), set_name)["scores"].astype(np.float64)


def load_micro(unit, set_name):
    return L.load_csv("04", "microstates", unit, state_params(unit), set_name)


def micro_seg(df):
    # Symmetrised copies get their own segment ids so no transition crosses the copies.
    seg = df["seg"].to_numpy(np.int64)
    return np.where(seg >= 0, 2 * seg + df["copy"].to_numpy(np.int64), -1)


def states_in_space(unit, data, seg, model, info):
    # States of `data` in a given state space (PCA model + bin edges). Nothing is refitted.
    use = seg >= 0
    if uses_physical(unit):
        pv = physical_vars(data["nodes"], unit)
        vals = [pv[v] for v in info["variables"]]
    else:
        feat, _ = unit_features(data["nodes"], unit, use)
        sc = S.pca_scores(feat, model)
        vals = [sc[:, int(v[2:]) - 1] for v in info["variables"]]
    bins = np.column_stack([S.digitize(np.where(use, v, np.nan), np.asarray(e)) for v, e in zip(vals, info["edges"])])
    n_use = max(int(use.sum()), 1)
    outside = {v: float(((b < 0) & use).sum() / n_use) for v, b in zip(info["variables"], bins.T)}
    return S.joint_state(bins, info["widths"]), outside


def state_names(info):
    return list(info["variables"])


print("output folder", L.root)
print("preprocessing label", S.param_label(pre_params()))
for _u in RUN_UNITS:
    print(f"  {_u:7s} state label {S.param_label(state_params(_u))}")

output folder \\bucket.oist.jp\bucket\StephensU\medaka\runs\behavior_features\outputs\20261107_pose_states
preprocessing label base
  FM1     state label bins10-pcs1_2_3
  FM2     state label bins10-pcs1_2_3
  M1M2    state label bins10-pcs1_2_3
  triple  state label bins10-pcs1_2_3


## Step 0 — 读数据、连续片段、缺口统计

**做什么**：读入每个视频的 F/M1/M2 三节点 3D 坐标；可选地做骨架阈值、短缺口插值、SG 平滑、体长归一化（默认都关）；对每个单元找出所需坐标全部有效的连续片段并记录起止帧；统计缺口。

**输入**：review package（`review_data.npz` + 身份审核账本）。

**输出**（`data/`）：`00_nodes__all__<pre>__K/P.npz`（每帧坐标、视频号、帧号、手动删除标记）、`00_segments__<unit>__<pre>__K/P.npz`（每帧的片段号，−1 = 不用）、`00_segments__<unit>__…__<set>.csv`（片段起止帧）、`00_gap_table__<unit>__…__<set>.csv`（仿 SI Table I）。图：`00_gap_hist`（缺口长度直方图，纵轴对数，"追踪丢失"和"手动删除"分开画）、`00_skeleton_dist`（仿 Fig. S3D，所有视频合并）。

后面所有步骤只用片段内部的帧，不跨缺口。`HEAD_PEC_MAX`、`PEC_TAIL_MAX` 是 None 时不画线、不剔除；看完直方图后手动填。

In [3]:
# Step 0: read data, optional preprocessing, continuous segments, gap statistics.
L.clear_log("00")
ALLOWED = set(K_EXPERIMENTS) | set(P_EXPERIMENTS)


def _allow_listed(experiment, final):
    # split.csv is not on this machine; only the videos named in the parameter cell are read.
    if experiment not in ALLOWED:
        raise PermissionError(f"{experiment} is not in K_EXPERIMENTS / P_EXPERIMENTS")


M.require_access = _allow_listed
PRE = pre_params()
GROUPS_NEEDED = [g for g in ("K", "P") if any(g in SET_GROUPS[s] for s in RUN_SETS)]
print("groups to read", GROUPS_NEEDED, " preprocessing label", S.param_label(PRE))

skel_hp, skel_pt = [], []
gap_rows = {u: [] for u in RUN_UNITS}
gap_runs = {u: [] for u in RUN_UNITS}
thr_rows = []
frame_counts = {"videos": {}, "valid_frames": {}}
REASONS = {1: "manually removed (review / ID pass)", 2: "tracking lost (accepted frame, NaN)",
           3: "pec XY too close (no X' direction)"}

for group in GROUPS_NEEDED:
    blocks = {"nodes": [], "video": [], "frame": [], "blocked": []}
    meta = {"names": [], "fps": [], "day": [], "bl_mm": []}
    valid_parts = {u: [] for u in RUN_UNITS}
    for vi, exp in enumerate(GROUP_EXPS[group]):
        t0 = time.time()
        trial = M.load_trial(exp, final=False)
        nodes = np.array(trial["named"], dtype=np.float64, copy=True)
        blocked = np.asarray(trial["blocked"], dtype=bool)
        frames = np.asarray(trial["frames"], dtype=np.int64)
        fps = float(trial["fps"])
        _grp, day = M.parse_experiment(exp)
        hp, pt = S.skeleton_distances(nodes)
        skel_hp.append(hp[np.isfinite(hp)])
        skel_pt.append(pt[np.isfinite(pt)])
        if HEAD_PEC_MAX is not None or PEC_TAIL_MAX is not None:
            bad = np.zeros(nodes.shape[:2], dtype=bool)
            if HEAD_PEC_MAX is not None:
                bad |= np.isfinite(hp) & (hp > HEAD_PEC_MAX)
            if PEC_TAIL_MAX is not None:
                bad |= np.isfinite(pt) & (pt > PEC_TAIL_MAX)
            nodes[bad] = np.nan
            n_ff = int((np.isfinite(hp) & np.isfinite(pt)).sum())
            thr_rows.append({"video": exp, "fish_frames_removed_percent": 100.0 * bad.sum() / max(n_ff, 1),
                             "frames_with_a_fish_removed_percent": 100.0 * bad.any(axis=1).mean()})
        if INTERPOLATE:
            nodes = M.fill_gaps(nodes, int(INTERP_MAX_FRAMES), blocked)
        if SMOOTH_SG:
            nodes = S.smooth_sg_keep_ends(nodes, int(SG_WINDOW), int(SG_ORDER))
        bl = S.body_lengths(nodes)
        # Validity is decided on millimetre coordinates (MIN_XY_MM is in mm).
        for unit in RUN_UNITS:
            finite_req = required_finite(nodes, unit)
            _pose, valid = unit_pose(nodes, unit, MIN_XY_MM)
            valid_parts[unit].append(valid)
            code = np.zeros(len(nodes), dtype=np.int8)
            code[blocked] = 1
            code[~blocked & ~finite_req] = 2
            code[~blocked & finite_req & ~valid] = 3
            g0, g1 = S.bool_runs(code > 0)
            gaps = g1 - g0
            s0, s1 = S.bool_runs(valid)
            gap_rows[unit].append({
                "group": group, "video": exp, "total_frames": len(nodes),
                "missing_percent": 100.0 * (code > 0).mean(),
                "mean_gap_frames": float(gaps.mean()) if len(gaps) else 0.0,
                "gap_p99_frames": float(np.percentile(gaps, 99)) if len(gaps) else 0.0,
                "n_segments": int(len(s0)),
                "longest_segment_s": float((s1 - s0).max() / fps) if len(s0) else 0.0,
                "manually_removed_percent": 100.0 * (code == 1).mean(),
                "tracking_lost_percent": 100.0 * (code == 2).mean(),
                "pec_xy_close_percent": 100.0 * (code == 3).mean(),
            })
            for r in REASONS:
                a, b = S.bool_runs(code == r)
                gap_runs[unit].append((group, r, b - a))
        if NORM_BL:
            if not np.all(np.isfinite(bl) & (bl > 0)):
                raise RuntimeError(f"{exp}: body length not usable {bl}")
            nodes = nodes / bl[None, :, None, None]
        blocks["nodes"].append(nodes.astype(np.float32))
        blocks["video"].append(np.full(len(nodes), vi, dtype=np.int16))
        blocks["frame"].append(frames)
        blocks["blocked"].append(blocked)
        meta["names"].append(exp)
        meta["fps"].append(fps)
        meta["day"].append(day)
        meta["bl_mm"].append(bl)
        finite_all = np.isfinite(nodes).all(axis=(1, 2, 3))
        frame_counts["videos"][exp] = {"frames": int(len(nodes)), "all_nodes_finite": int(finite_all.sum()),
                                       "blocked": int(blocked.sum())}
        print(f"{exp}  frames {len(nodes)} ({frames[0]}..{frames[-1]})  shape {nodes.shape}  fps {fps:.3f}  "
              f"all-finite {int(finite_all.sum())}  manually removed {int(blocked.sum())}  "
              f"body length mm {np.round(bl, 2)}  coord range {np.nanmin(nodes):.2f}..{np.nanmax(nodes):.2f} "
              f"{coord_unit()}  ({time.time() - t0:.0f} s)")
    video = np.concatenate(blocks["video"])
    L.save_npz({"nodes": np.concatenate(blocks["nodes"]), "video": video,
                "frame": np.concatenate(blocks["frame"]), "blocked": np.concatenate(blocks["blocked"]),
                "names": np.array(meta["names"]), "fps": np.array(meta["fps"]), "day": np.array(meta["day"]),
                "body_length_mm": np.array(meta["bl_mm"])}, "00", "nodes", "all", PRE, group)
    for unit in RUN_UNITS:
        valid = np.concatenate(valid_parts[unit])
        seg = S.segment_ids(valid, video)
        L.save_npz({"seg": seg.astype(np.int32)}, "00", "segments", unit, seg_params(unit), group, verbose=False)
        print(f"  {group} {unit:7s} valid frames {int(valid.sum())} / {len(valid)}  segments {int(seg.max()) + 1}")
_CACHE.clear()

# --- skeleton-distance histograms (raw mm, all videos pooled), thresholds only if given
hp_all, pt_all = np.concatenate(skel_hp), np.concatenate(skel_pt)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, data, thr, label in ((axes[0], hp_all, HEAD_PEC_MAX, "head-pec"), (axes[1], pt_all, PEC_TAIL_MAX, "pec-tail")):
    hi = float(np.percentile(data, 99.95)) * 1.3
    ax.hist(data[data <= hi], bins=150, color="0.35")
    ax.set_yscale("log")
    ax.set_xlabel(f"{label} distance (mm, raw)")
    ax.set_ylabel("fish x frames")
    outside = 100.0 * (data > hi).mean()
    txt = f"n={len(data)}; {outside:.3f}% beyond {hi:.1f} mm not shown"
    if thr is not None:
        ax.axvline(thr, color="k", ls="--", lw=1)
        txt += f"\nthreshold {thr} mm: {100.0 * (data > thr).mean():.3f}% of fish-frames above"
    ax.set_title(txt, fontsize=9)
fig.suptitle("Skeleton distances, all videos and fish pooled (thresholds are set by hand, none drawn if None)")
save_fig(fig, "00", "skeleton_dist", "all", PRE, "all")
print(f"head-pec mm: median {np.median(hp_all):.2f}  p0.1 {np.percentile(hp_all, 0.1):.2f}  p99.9 {np.percentile(hp_all, 99.9):.2f}")
print(f"pec-tail mm: median {np.median(pt_all):.2f}  p0.1 {np.percentile(pt_all, 0.1):.2f}  p99.9 {np.percentile(pt_all, 99.9):.2f}")
if thr_rows:
    thr_df = pd.DataFrame(thr_rows)
    L.save_csv(thr_df, "00", "skeleton_thresh_removed", "all", PRE, "all")
    print(thr_df.to_string(index=False))
else:
    print("HEAD_PEC_MAX / PEC_TAIL_MAX are None: no line drawn, no frame removed")

# --- per analysis set: segment tables, gap tables, gap-length histograms
for set_name in RUN_SETS:
    groups = SET_GROUPS[set_name]
    data = load_set(set_name)
    for unit in RUN_UNITS:
        seg = load_seg(unit, set_name)
        table = S.segment_table(seg, data["names"][data["video"]], data["frame"], data["fps"])
        L.save_csv(table, "00", "segments", unit, seg_params(unit), set_name, verbose=False)
        gaps = pd.DataFrame([r for r in gap_rows[unit] if r["group"] in groups]).drop(columns="group")
        L.save_csv(gaps, "00", "gap_table", unit, seg_params(unit), set_name, verbose=False)
        frame_counts["valid_frames"][f"{set_name}/{unit}"] = int((seg >= 0).sum())
        print(f"\n[{set_name} {unit_title(unit)}] frames {len(seg)}  valid {int((seg >= 0).sum())}  "
              f"segments {len(table)}  length median {table['n_frames'].median():.0f} max {table['n_frames'].max()} frames")
        print(gaps.round(3).to_string(index=False))
        fig, ax = plt.subplots(figsize=(7, 4))
        all_len = [l for g, r, l in gap_runs[unit] if g in groups]
        top = max([int(l.max()) for l in all_len if len(l)] + [2])
        bins = np.unique(np.round(np.logspace(0, np.log10(top + 1), 60)).astype(int))
        for r, color in ((1, "#7f7f7f"), (2, "#d62728"), (3, "#1f77b4")):
            lens = np.concatenate([l for g, rr, l in gap_runs[unit] if g in groups and rr == r] + [np.zeros(0, int)])
            if len(lens):
                ax.hist(lens, bins=bins, histtype="step", lw=1.5, color=color, label=f"{REASONS[r]} (n={len(lens)})")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("gap length (frames; log-spaced bins)")
        ax.set_ylabel("number of gaps")
        ax.legend(fontsize=8)
        ax.set_title(f"{set_name} {unit_title(unit)}: gap lengths by cause")
        save_fig(fig, "00", "gap_hist", unit, seg_params(unit), set_name)
L.update_params("frame_counts", frame_counts)
L.log("00", "done")

groups to read ['K', 'P']  preprocessing label base


20251225_K_1_1_FCMC  frames 348600 (0..348599)  shape (348600, 3, 3, 3)  fps 166.000  all-finite 330760  manually removed 11893  body length mm [24.63 24.37 25.23]  coord range -30.75..212.44 mm  (1 s)


20251225_K_1_2_FCMC  frames 348600 (0..348599)  shape (348600, 3, 3, 3)  fps 166.000  all-finite 323710  manually removed 20826  body length mm [24.77 24.45 25.31]  coord range -10.81..196.43 mm  (1 s)


  npz data/00_nodes__all__base__K.npz
  K FM1     valid frames 658920 / 697200  segments 1520
  K FM2     valid frames 657010 / 697200  segments 1576


  K M1M2    valid frames 656056 / 697200  segments 1774
  K triple  valid frames 654470 / 697200  segments 2174


20260112_P_1_1_FCMC  frames 348600 (0..348599)  shape (348600, 3, 3, 3)  fps 166.000  all-finite 321996  manually removed 18355  body length mm [23.77 25.82 25.44]  coord range -24.36..204.10 mm  (1 s)


... (14 outputs left out)

  fig figures/00_gap_hist__FM1__base__KP.png

[KP FM2 (F->M2)] frames 1394400  valid 1312685  segments 3408  length median 9 max 14609 frames
              video  total_frames  missing_percent  mean_gap_frames  gap_p99_frames  n_segments  longest_segment_s  manually_removed_percent  tracking_lost_percent  pec_xy_close_percent
20251225_K_1_1_FCMC        348600            4.688           19.739          274.73         827             68.355                     3.412                  1.168                 0.109
20251225_K_1_2_FCMC        348600            6.841           31.795          236.36         749             58.307                     5.974                  0.775                 0.091
20260112_P_1_1_FCMC        348600            6.919           30.185          213.12         798             77.566                     5.265                  1.619                 0.034
20260113_P_1_2_FCMC        348600            4.993           16.818          167.00        1034             88.006

  fig figures/00_gap_hist__FM2__base__KP.png

[KP M1M2 (M1->M2)] frames 1394400  valid 1314346  segments 3208  length median 9 max 12149 frames
              video  total_frames  missing_percent  mean_gap_frames  gap_p99_frames  n_segments  longest_segment_s  manually_removed_percent  tracking_lost_percent  pec_xy_close_percent
20251225_K_1_1_FCMC        348600            4.931           17.776          262.12         966             63.807                     3.412                  1.478                 0.041
20251225_K_1_2_FCMC        348600            6.872           29.611          217.76         808             45.277                     5.974                  0.879                 0.018
20260112_P_1_1_FCMC        348600            6.341           32.552          226.44         678             69.783                     5.265                  1.051                 0.024
20260113_P_1_2_FCMC        348600            4.821           22.202          167.00         756             73.1

  fig figures/00_gap_hist__M1M2__base__KP.png

[KP triple (raw coordinates)] frames 1394400  valid 1305220  segments 4590  length median 7 max 11584 frames
              video  total_frames  missing_percent  mean_gap_frames  gap_p99_frames  n_segments  longest_segment_s  manually_removed_percent  tracking_lost_percent  pec_xy_close_percent
20251225_K_1_1_FCMC        348600            5.118           15.406          214.74        1157             57.922                     3.412                  1.706                   0.0
20251225_K_1_2_FCMC        348600            7.140           24.450          167.00        1017             52.331                     5.974                  1.166                   0.0
20260112_P_1_1_FCMC        348600            7.632           27.455          228.56         968             69.783                     5.265                  2.366                   0.0
20260113_P_1_2_FCMC        348600            5.693           13.696          123.92        1448     

  fig figures/00_gap_hist__triple__base__KP.png


## Step 1 — PCA

**版本 A（配对）**：每帧平移到两条鱼 pec 的中点；Z' = Z；X' = 鱼1 pec → 鱼2 pec 在 XY 平面的投影方向；Y' 补全右手系。旋转后两个 pec 的 y' 恒为 0，x'、z' 大小相等符号相反。PCA 输入 14 个量（见假设 6）；鱼2 的 pec 是镜像，不进 PCA。打印两个 pec 的 max|y'|、max|x'₁+x'₂|（应 ≈ 0）和输入形状（T×14）。

**版本 B（triple）**：27 个原始坐标，不变换（`TRIPLE_COROTATE=False` 时）。

**输出**：`01_cum_var`（累积方差条形图，仿 Fig. 2a；颜色由 `PC_GROUPS` 决定，默认单色）、`01_loadings`（载荷热图，仿 Fig. S5；列下方括号标 `PC_GROUPS`）；`data/` 里有 `01_pca_model`（均值、尺度、载荷矩阵，供 Step 6）、`01_pc_scores`（每帧 PC 得分，片段外为 NaN）、`01_loadings.csv`、`01_loadings_above_thr.csv`（|载荷| > `LOAD_THR`）、`01_explained_variance.csv`。

`PC_GROUPS` 由你看完载荷热图后手动填，程序不会自动分组。

In [4]:
# Step 1: PCA per analysis set and unit.
L.clear_log("01")


def pc_group_colors(n):
    if not PC_GROUPS:
        return ["#4C78A8"] * n, []
    color_of = {}
    handles = []
    for i, (gname, pcs) in enumerate(PC_GROUPS.items()):
        c = PALETTE[i % len(PALETTE)]
        handles.append(plt.Rectangle((0, 0), 1, 1, color=c, label=gname))
        for pc in pcs:
            color_of[int(pc)] = c
    return [color_of.get(i + 1, "#bdbdbd") for i in range(n)], handles


def contiguous(pcs):
    pcs = sorted(int(p) for p in pcs)
    runs, start = [], pcs[0]
    for a, b in zip(pcs, pcs[1:] + [None]):
        if b != a + 1:
            runs.append((start, a))
            start = b
    return runs


for set_name in RUN_SETS:
    data = load_set(set_name)
    for unit in RUN_UNITS:
        p1 = unit_params(unit)
        seg = load_seg(unit, set_name)
        use = seg >= 0
        feat, pose = unit_features(data["nodes"], unit, use)
        if not np.isfinite(feat[use]).all():
            raise RuntimeError(f"{set_name} {unit}: non-finite features inside segments")
        x = feat[use]
        names = feature_names(unit)
        print(f"\n[{set_name} {unit_title(unit)}] PCA input {x.shape} (expected T x {len(names)})  "
              f"value range {x.min():.3f}..{x.max():.3f} {coord_unit()}")
        if is_pair(unit):
            pu = pose[use]
            print(f"  max|pec y'| fish1 {np.abs(pu[:, 1, 1]).max():.2e}  fish2 {np.abs(pu[:, 4, 1]).max():.2e}  "
                  f"max|pec x'1 + x'2| {np.abs(pu[:, 1, 0] + pu[:, 4, 0]).max():.2e}  "
                  f"max|pec z'1 + z'2| {np.abs(pu[:, 1, 2] + pu[:, 4, 2]).max():.2e}")
        model = S.fit_pca(x, PCA_STANDARDIZE, FIX_PC_SIGN)
        scores = S.pca_scores(feat, model)
        expl = model["explained"]
        cum = 100.0 * np.cumsum(expl)
        n90 = int(np.searchsorted(cum, 90.0) + 1)
        print(f"  variance % first 6 {np.round(100 * expl[:6], 2)}  PCs for 90%: {n90}  flipped {int(model['flipped'].sum())}")
        L.save_npz({"mean": model["mean"], "scale": model["scale"], "components": model["components"],
                    "eigvals": model["eigvals"], "explained": expl, "flipped": model["flipped"],
                    "feature_names": np.array(names)}, "01", "pca_model", unit, p1, set_name)
        L.save_npz({"scores": scores.astype(np.float32)}, "01", "pc_scores", unit, p1, set_name)
        n_pc = len(expl)
        pcs = [f"PC{i + 1}" for i in range(n_pc)]
        L.save_csv(pd.DataFrame({"pc": np.arange(1, n_pc + 1), "explained_percent": 100 * expl,
                                 "cumulative_percent": cum, "eigenvalue": model["eigvals"]}),
                   "01", "explained_variance", unit, p1, set_name, verbose=False)
        comp = model["components"]
        L.save_csv(pd.DataFrame(comp, columns=pcs).assign(feature=names)[["feature"] + pcs],
                   "01", "loadings", unit, p1, set_name, verbose=False)
        above = [{"pc": j + 1, "feature": names[i], "loading": comp[i, j]}
                 for j in range(n_pc) for i in np.argsort(-np.abs(comp[:, j])) if abs(comp[i, j]) > LOAD_THR]
        L.save_csv(pd.DataFrame(above, columns=["pc", "feature", "loading"]), "01", "loadings_above_thr",
                   unit, p1, set_name, verbose=False)
        print(f"  loadings with |value| > {LOAD_THR}: {len(above)}")

        colors, handles = pc_group_colors(n_pc)
        fig, ax = plt.subplots(figsize=(max(5.0, 0.32 * n_pc + 2), 3.8))
        ax.bar(np.arange(1, n_pc + 1), cum, color=colors, width=0.8)
        ymin = CUMVAR_YMIN if cum[0] >= CUMVAR_YMIN else max(0.0, 5 * np.floor(cum[0] / 5) - 5)
        ax.set_ylim(ymin, 100.5)
        ax.set_xticks(np.arange(1, n_pc + 1))
        ax.tick_params(axis="x", labelsize=7)
        ax.set_xlabel("PC")
        ax.set_ylabel("cumulative explained variance (%)")
        if handles:
            ax.legend(handles=handles, fontsize=8, loc="lower right")
        ax.set_title(f"{set_name} {unit_title(unit)}: cumulative variance (PCs for 90%: {n90})")
        save_fig(fig, "01", "cum_var", unit, p1, set_name)

        vmax = float(np.abs(comp).max())
        fig, ax = plt.subplots(figsize=(max(6.0, 0.36 * n_pc + 3.5), max(4.0, 0.3 * len(names) + 2)))
        im = ax.imshow(comp, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto", interpolation="nearest")
        ax.set_yticks(np.arange(len(names)), names, fontsize=8)
        ax.set_xticks(np.arange(n_pc), pcs, rotation=90, fontsize=8)
        ax.xaxis.tick_top()
        for gname, members in PC_GROUPS.items():
            for a, b in contiguous(members):
                if a > n_pc:
                    continue
                b = min(b, n_pc)
                y0 = len(names) - 0.5 + 0.3
                ax.plot([a - 1.4, a - 1.4, b - 0.6, b - 0.6], [y0, y0 + 0.4, y0 + 0.4, y0], color="k", lw=1, clip_on=False)
                ax.text((a + b) / 2 - 1, y0 + 0.6, gname, ha="center", va="top", fontsize=8, clip_on=False)
        ax.set_ylim(len(names) - 0.5, -0.5)
        fig.colorbar(im, ax=ax, fraction=0.04, label="eigenvector component")
        ax.set_title(f"{set_name} {unit_title(unit)}: PCA loadings (sign rule FIX_PC_SIGN={FIX_PC_SIGN}, "
                     f"standardised={PCA_STANDARDIZE})", fontsize=10, pad=28)
        save_fig(fig, "01", "loadings", unit, p1, set_name)
L.log("01", "done")


[K FM1 (F->M1)] PCA input (658920, 14) (expected T x 14)  value range -128.177..126.545 mm
  max|pec y'| fish1 1.42e-14  fish2 1.42e-14  max|pec x'1 + x'2| 0.00e+00  max|pec z'1 + z'2| 0.00e+00
  variance % first 6 [61.92 33.37  1.27  1.04  0.98  0.62]  PCs for 90%: 2  flipped 7
  npz data/01_pca_model__FM1__base__K.npz


  npz data/01_pc_scores__FM1__base__K.npz
  loadings with |value| > 0.3: 40
  fig figures/01_cum_var__FM1__base__K.png


  fig figures/01_loadings__FM1__base__K.png



[K FM2 (F->M2)] PCA input (657010, 14) (expected T x 14)  value range -128.614..127.750 mm
  max|pec y'| fish1 1.42e-14  fish2 1.42e-14  max|pec x'1 + x'2| 0.00e+00  max|pec z'1 + z'2| 0.00e+00
  variance % first 6 [54.66 36.55  2.54  2.4   1.48  1.12]  PCs for 90%: 2  flipped 10
  npz data/01_pca_model__FM2__base__K.npz


  npz data/01_pc_scores__FM2__base__K.npz
  loadings with |value| > 0.3: 44
  fig figures/01_cum_var__FM2__base__K.png


  fig figures/01_loadings__FM2__base__K.png


... (33 outputs left out)


[KP triple (raw coordinates)] PCA input (1305220, 27) (expected T x 27)  value range -38.917..212.435 mm


  variance % first 6 [22.4  16.2  14.78  9.67  9.24  7.75]  PCs for 90%: 8  flipped 16
  npz data/01_pca_model__triple__base__KP.npz


  npz data/01_pc_scores__triple__base__KP.npz
  loadings with |value| > 0.3: 101
  fig figures/01_cum_var__triple__base__KP.png


  fig figures/01_loadings__triple__base__KP.png


## Step 2 — 把主成分"画出来"

**做什么**：对前 `N_SHOW` 个 PC，骨架(k) = 平均骨架 + k·σ_PC·载荷向量，k = −2, 0, +2（σ_PC 是该 PC 得分的标准差，其余 PC 取平均）。载荷向量还原成 鱼×点×坐标；配对版本里鱼2 的 pec 由镜像补出。三个 k 画在同一张图（线型 `:` = −2，`-` = 0，`--` = +2），XY、XZ 两个视图，每条鱼一种颜色并标 F/M1/M2。PC 的符号是任意的。

**输出**：图 `02_skeleton_pc<j>`；`data/02_pc_physical_corr.csv`（各 PC 得分与物理量的 Pearson 相关：pec–pec 距离、每条鱼的体弯曲角 head–pec–tail、每条鱼 pec→head 在 XY 平面相对 X' 轴的朝向角；版本 B 另加整体质心 x、y、z 和每条鱼 pec 的 x、y、z）；版本 B 另有 `02_loading_xyz_share.csv`（每个 PC 的 |载荷| 按 x、y、z 汇总的占比）。

In [5]:
# Step 2: reconstruct skeletons along each PC, correlate PC scores with physical quantities.
L.clear_log("02")


def k_style(k):
    return (":" if k < 0 else "--" if k > 0 else "-"), (1.0 if k == 0 else 0.7)


def skeleton_points(x, unit):
    if is_pair(unit):
        pts = S.pair_pose_from_features(x)[0]
        a, b = S.PAIR_UNITS[unit]
        return {a: pts[0:3], b: pts[3:6]}
    pts = np.asarray(x).reshape(3, 3, 3)
    return {f: pts[i] for i, f in enumerate(S.FISH)}


def corr_quantities(nodes, unit, pose):
    q = {}
    tick = prime(unit)
    if is_pair(unit):
        a, b = S.PAIR_UNITS[unit]
        ia, ib = S.FISH.index(a), S.FISH.index(b)
        q["pec_pec_dist"] = np.linalg.norm(nodes[:, ib, S.PEC] - nodes[:, ia, S.PEC], axis=1)
        for f, i in ((a, ia), (b, ib)):
            q[f"bend_deg_{f}"] = bend_deg(nodes, i)
        for f, off in ((a, 0), (b, 3)):
            h = pose[:, off + 0] - pose[:, off + 1]
            q[f"heading_deg_{f}_vs_X'"] = np.degrees(np.arctan2(h[:, 1], h[:, 0]))
        return q
    for pu in ("FM1", "FM2", "M1M2"):
        a, b = S.PAIR_UNITS[pu]
        q[f"pec_pec_dist_{a}{b}"] = np.linalg.norm(nodes[:, S.FISH.index(b), S.PEC] - nodes[:, S.FISH.index(a), S.PEC], axis=1)
    for i, f in enumerate(S.FISH):
        q[f"bend_deg_{f}"] = bend_deg(nodes, i)
    p = pose.reshape(len(pose), 3, 3, 3)
    for i, f in enumerate(S.FISH):
        h = p[:, i, S.HEAD] - p[:, i, S.PEC]
        q[f"heading_deg_{f}_vs_X{tick}"] = np.degrees(np.arctan2(h[:, 1], h[:, 0]))
    cen = p.reshape(len(p), 9, 3).mean(axis=1)
    for j, axn in enumerate("xyz"):
        q[f"centroid_{axn}{tick}"] = cen[:, j]
    for i, f in enumerate(S.FISH):
        for j, axn in enumerate("xyz"):
            q[f"{f}_pec_{axn}{tick}"] = p[:, i, S.PEC, j]
    return q


def corr_table(scores, quantities, use):
    out = {}
    for name, v in quantities.items():
        m = use & np.isfinite(v)
        s = scores[m]
        z = v[m]
        sd = s.std(axis=0)
        sd[sd == 0] = np.nan
        out[name] = ((s - s.mean(axis=0)) * (z - z.mean())[:, None]).mean(axis=0) / (sd * z.std())
    return pd.DataFrame(out)


for set_name in RUN_SETS:
    data = load_set(set_name)
    for unit in RUN_UNITS:
        p1 = unit_params(unit)
        model = load_model(unit, set_name)
        scores = load_scores(unit, set_name)
        seg = load_seg(unit, set_name)
        use = seg >= 0
        sigma = np.nanstd(scores[use], axis=0, ddof=1)
        tick = prime(unit)
        cu = coord_unit()
        print(f"\n[{set_name} {unit_title(unit)}] frames used {int(use.sum())}  sigma(PC1..{N_SHOW}) "
              f"{np.round(sigma[:N_SHOW], 3)} {cu}")
        for j in range(min(N_SHOW, len(sigma))):
            fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
            for k in K_SIGMAS:
                x = model["mean"] + model["scale"] * (k * sigma[j] * model["components"][:, j])
                ls, alpha = k_style(k)
                for fish, pts in skeleton_points(x, unit).items():
                    for ax, (i0, i1) in zip(axes, ((0, 1), (0, 2))):
                        ax.plot(pts[:, i0], pts[:, i1], ls=ls, color=FISH_COLOR[fish], alpha=alpha, lw=2)
                        ax.plot(pts[0, i0], pts[0, i1], "o", color=FISH_COLOR[fish], alpha=alpha, ms=5)
                        if k == 0:
                            ax.text(pts[0, i0], pts[0, i1], f" {fish}", color=FISH_COLOR[fish], fontsize=10, va="bottom")
            for ax, (lx, ly) in zip(axes, (("x", "y"), ("x", "z"))):
                ax.set_aspect("equal", adjustable="datalim")
                ax.set_xlabel(f"{lx}{tick} ({cu})")
                ax.set_ylabel(f"{ly}{tick} ({cu})")
                ax.grid(alpha=0.3)
            axes[0].set_title("XY view (top)", fontsize=9)
            axes[1].set_title("XZ view (side)", fontsize=9)
            handles = [Line2D([], [], color="k", ls=k_style(k)[0], label=f"k = {k:+d}") for k in K_SIGMAS]
            handles += [Line2D([], [], color="k", marker="o", ls="", label="head")]
            axes[1].legend(handles=handles, fontsize=8, loc="best")
            fig.suptitle(f"{set_name} {unit_title(unit)}  PC{j + 1} ({100 * model['explained'][j]:.1f}% var): "
                         f"mean + k*sigma*loading, sigma = {sigma[j]:.2f} {cu}\n"
                         f"PC sign is arbitrary (FIX_PC_SIGN={FIX_PC_SIGN}); other PCs at their mean", fontsize=10)
            if not is_pair(unit):
                fig.text(0.01, -0.02, "Note: the mean skeleton is an average over different orientations; it can be "
                                      "shortened or distorted. For reference only.", fontsize=9)
            save_fig(fig, "02", f"skeleton_pc{j + 1}", unit, p1, set_name)
        feat, pose = unit_features(data["nodes"], unit, use)
        q = corr_quantities(data["nodes"], unit, pose)
        ct = corr_table(scores, q, use)
        ct.insert(0, "pc", np.arange(1, len(ct) + 1))
        L.save_csv(ct, "02", "pc_physical_corr", unit, p1, set_name, verbose=False)
        top = ct.head(min(5, len(ct))).set_index("pc").round(2)
        print("  correlations, first PCs:\n" + top.to_string())
        if not is_pair(unit):
            comp = np.abs(model["components"])
            names = feature_names(unit)
            share = {f"share_{a}{tick}": comp[[n.endswith(f"_{a}{tick}") for n in names]].sum(axis=0) / comp.sum(axis=0)
                     for a in "xyz"}
            sh = pd.DataFrame(share)
            sh.insert(0, "pc", np.arange(1, len(sh) + 1))
            L.save_csv(sh, "02", "loading_xyz_share", unit, p1, set_name, verbose=False)
            print("  |loading| share by axis, first 6 PCs:\n" + sh.head(6).round(3).to_string(index=False))
L.log("02", "done")


[K FM1 (F->M1)] frames used 658920  sigma(PC1..3) [90.79  66.655 12.983] mm


  fig figures/02_skeleton_pc1__FM1__base__K.png
  fig figures/02_skeleton_pc2__FM1__base__K.png


  fig figures/02_skeleton_pc3__FM1__base__K.png


  correlations, first PCs:
    pec_pec_dist  bend_deg_F  bend_deg_M1  heading_deg_F_vs_X'  heading_deg_M1_vs_X'
pc                                                                                  
1           0.44        0.07        -0.10                -0.03                  0.01
2           0.80       -0.01        -0.08                 0.05                  0.03
3          -0.01       -0.02         0.00                -0.64                 -0.49
4          -0.04       -0.03         0.01                 0.13                 -0.11
5           0.02        0.01        -0.03                 0.39                 -0.54



[K FM2 (F->M2)] frames used 657010  sigma(PC1..3) [64.911 53.081 14.006] mm


  fig figures/02_skeleton_pc1__FM2__base__K.png


... (44 outputs left out)

  fig figures/02_skeleton_pc1__triple__base__KP.png


  fig figures/02_skeleton_pc2__triple__base__KP.png


  fig figures/02_skeleton_pc3__triple__base__KP.png


  correlations, first PCs:
    pec_pec_dist_FM1  pec_pec_dist_FM2  pec_pec_dist_M1M2  bend_deg_F  bend_deg_M1  bend_deg_M2  heading_deg_F_vs_X  heading_deg_M1_vs_X  heading_deg_M2_vs_X  centroid_x  centroid_y  centroid_z  F_pec_x  F_pec_y  F_pec_z  M1_pec_x  M1_pec_y  M1_pec_z  M2_pec_x  M2_pec_y  M2_pec_z
pc                                                                                                                                                                                                                                                                                      
1               0.18             -0.15               0.02        0.07        -0.04        -0.05               -0.14                -0.04                -0.05        0.22       -0.18        0.97     0.12    -0.26     0.84      0.19     -0.09      0.64      0.16     -0.02      0.74
2              -0.06             -0.01              -0.07       -0.03         0.03        -0.02                0.05               

## Step 3 — 自相关函数（ACF）

**做什么**：每个 PC 得分标准化为单位方差；在每个连续片段内部算自相关，按片段长度加权平均（见假设 10）。最大滞后 `MAX_LAG_S` 秒，横轴用秒；画前 `N_ACF` 个 PC，并标出降到 1/e 的时间。

**输出**：图 `03_acf`；`data/03_acf.npz`（滞后、曲线、每个滞后的权重）；`data/03_acf_times.csv`（每个 PC 的 1/e 相关时间，秒和帧；空 = 在 `MAX_LAG_S` 内没有降到 1/e）。

In [6]:
# Step 3: autocorrelation of the PC scores inside continuous segments.
L.clear_log("03")
for set_name in RUN_SETS:
    data = load_set(set_name)
    fps = set_fps(data)
    max_lag = int(round(MAX_LAG_S * fps))
    lag_s = np.arange(max_lag + 1) / fps
    for unit in RUN_UNITS:
        p1 = unit_params(unit)
        scores = load_scores(unit, set_name)
        seg = load_seg(unit, set_name)
        use = seg >= 0
        n_pc = min(max(N_ACF, N_CMP), scores.shape[1])
        s = scores[:, :n_pc]
        mu = s[use].mean(axis=0)
        sd = s[use].std(axis=0)
        sd[sd == 0] = 1.0
        z = np.where(use[:, None], (s - mu) / sd, 0.0)
        curve, weight = S.acf_segments(z, seg, max_lag)
        times = np.array([S.time_to_e(lag_s, curve[:, j]) for j in range(n_pc)])
        L.save_npz({"lag_s": lag_s, "curve": curve, "weight": weight}, "03", "acf", unit, p1, set_name, verbose=False)
        tab = pd.DataFrame({"pc": np.arange(1, n_pc + 1), "tau_1e_s": times, "tau_1e_frames": times * fps})
        L.save_csv(tab, "03", "acf_times", unit, p1, set_name, verbose=False)
        print(f"[{set_name} {unit_title(unit)}] lag 0 = {np.round(curve[0, :3], 4)}  frames at max lag "
              f"{int(weight[-1])}/{int(weight[0])}  1/e (s) PC1..{min(n_pc, 6)}: {np.round(times[:6], 3)}")
        fig, ax = plt.subplots(figsize=(8, 4.5))
        cmap = plt.get_cmap("viridis")
        n_show = min(N_ACF, n_pc)
        for j in range(n_show):
            c = cmap(j / max(n_show - 1, 1))
            ax.plot(lag_s, curve[:, j], color=c, lw=1.4, label=f"PC{j + 1}  1/e {times[j]:.2f} s" if np.isfinite(times[j])
                    else f"PC{j + 1}  1/e > {MAX_LAG_S:g} s")
            if np.isfinite(times[j]):
                ax.plot(times[j], np.exp(-1), "o", color=c, ms=5)
        ax.axhline(np.exp(-1), color="k", ls="--", lw=0.8)
        ax.axhline(0, color="0.5", lw=0.5)
        ax.set_xlabel("lag (s)")
        ax.set_ylabel("autocorrelation")
        ax.set_xlim(0, MAX_LAG_S)
        ax.legend(fontsize=7, ncol=2)
        ax.set_title(f"{set_name} {unit_title(unit)}: ACF of standardised PC scores (within segments, "
                     f"length-weighted); dashed = 1/e")
        save_fig(fig, "03", "acf", unit, p1, set_name)
L.log("03", "done")

[K FM1 (F->M1)] lag 0 = [1.0001 0.9996 1.    ]  frames at max lag 555663/658615  1/e (s) PC1..6: [  nan 4.996 2.065 1.834 1.199 1.003]


  fig figures/03_acf__FM1__base__K.png


[K FM2 (F->M2)] lag 0 = [1.0003 1.0002 0.9998]  frames at max lag 549359/656703  1/e (s) PC1..6: [  nan 2.935 1.66  1.283 1.114 0.972]


  fig figures/03_acf__FM2__base__K.png


[K M1M2 (M1->M2)] lag 0 = [0.9999 1.     1.0001]  frames at max lag 546935/655668  1/e (s) PC1..6: [3.874 2.761 1.166 1.173 0.954 0.782]


  fig figures/03_acf__M1M2__base__K.png


... (10 outputs left out)

  fig figures/03_acf__FM1__base__KP.png


[KP FM2 (F->M2)] lag 0 = [1.     1.0001 1.    ]  frames at max lag 1085034/1311968  1/e (s) PC1..6: [  nan   nan 2.155 1.848 1.82  1.249]
  fig figures/03_acf__FM2__base__KP.png


[KP M1M2 (M1->M2)] lag 0 = [1.0001 0.9999 1.    ]  frames at max lag 1123239/1313665  1/e (s) PC1..6: [  nan 3.978 1.757 1.807 1.415 0.967]
  fig figures/03_acf__M1M2__base__KP.png


[KP triple (raw coordinates)] lag 0 = [0.9995 0.9997 0.9995]  frames at max lag 1026493/1304196  1/e (s) PC1..6: [nan nan nan nan nan nan]
  fig figures/03_acf__triple__base__KP.png


## Step 4 — 离散化（微状态）

**做什么**：`STATE_VARS="pcs"`：对 `PCS_FOR_STATE` 指定的 PC 得分分格；`"physical"`（只用于版本 A）：D_PP、θ₁、θ₂（见假设 12、13）。triple 总是用 PC。`N_BINS` 格的边界由当前分析集所有有效帧的最小值到最大值等宽切分（`BIN_BY_PERCENTILE=True` 时按 `PERCENTILE` 切分）。边界保存下来，两个 τ 共用。

两种模式：`joint`（联合成一个状态，N_BINS^n 个状态）、`per_pc`（每个变量单独离散化）。

**输出**：`data/04_bin_edges.json`、`data/04_microstates.csv`（视频、帧号、片段号、副本标记、各变量格子编号、joint 状态编号；只含片段内的帧；−1 = 超出范围）；图 `04_occupancy_joint`、`04_occupancy_per_pc`；打印被占用状态数/总状态数。

In [7]:
# Step 4: bin the state variables; joint and per-variable states.
L.clear_log("04")
if STATE_VARS == "physical" and "triple" in RUN_UNITS:
    print("STATE_VARS='physical' is defined for pairs only; triple uses PC scores")
for set_name in RUN_SETS:
    data = load_set(set_name)
    fps = set_fps(data)
    vnames = data["names"]
    for unit in RUN_UNITS:
        sp = state_params(unit)
        seg = load_seg(unit, set_name)
        use = seg >= 0
        n_use = int(use.sum())
        if uses_physical(unit):
            pv = physical_vars(data["nodes"], unit)
            var_names = list(pv)
            values = [pv[k] for k in var_names]
            edges = [np.linspace(0.0, D_MAX, N_BINS_D + 1)] + [np.linspace(-np.pi, np.pi, N_BINS_THETA + 1)] * 2
            value_units = [coord_unit(), "rad", "rad"]
        else:
            scores = load_scores(unit, set_name)
            var_names = [f"PC{p}" for p in PCS_FOR_STATE]
            values = [scores[:, p - 1] for p in PCS_FOR_STATE]
            pct = tuple(PERCENTILE) if BIN_BY_PERCENTILE else None
            edges = [S.equal_edges(v[use], N_BINS, pct) for v in values]
            value_units = ["PC score"] * len(values)
        widths = [len(e) - 1 for e in edges]
        n_states = int(np.prod(widths))
        print(f"\n[{set_name} {unit_title(unit)}] variables {var_names}  frames in segments {n_use}")
        bins = np.column_stack([S.digitize(np.where(use, v, np.nan), e) for v, e in zip(values, edges)])
        for name, v, b, e in zip(var_names, values, bins.T, edges):
            out = int(((b < 0) & use).sum())
            print(f"  {name}: values {np.nanmin(v[use]):.3f}..{np.nanmax(v[use]):.3f}  edges {e[0]:.3f}..{e[-1]:.3f}  "
                  f"outside / NaN {out} ({100.0 * out / max(n_use, 1):.3f}%)")
        state = S.joint_state(bins, widths)
        idx = np.flatnonzero(use)
        cols = {"video": vnames[data["video"][idx]], "frame": data["frame"][idx], "seg": seg[idx],
                "copy": np.zeros(len(idx), dtype=np.int8)}
        for name, b in zip(var_names, bins.T):
            cols[f"bin_{name}"] = b[idx]
        cols["state"] = state[idx]
        table = pd.DataFrame(cols)
        if sp.get("sym"):
            swapped = bins[:, [0, 2, 1]]
            cols2 = dict(cols)
            cols2["copy"] = np.ones(len(idx), dtype=np.int8)
            for name, b in zip(var_names, swapped.T):
                cols2[f"bin_{name}"] = b[idx]
            cols2["state"] = S.joint_state(swapped, widths)[idx]
            table = pd.concat([table, pd.DataFrame(cols2)], ignore_index=True)
            print(f"  SYMMETRIZE_M1M2: added {len(idx)} swapped copies")
        st = table["state"].to_numpy()
        occ = np.bincount(st[st >= 0], minlength=n_states)
        occ_var = [int((np.bincount(table[f"bin_{nm}"].to_numpy()[table[f"bin_{nm}"].to_numpy() >= 0],
                                    minlength=w) > 0).sum()) for nm, w in zip(var_names, widths)]
        L.save_csv(table, "04", "microstates", unit, sp, set_name)
        L.save_json({"variables": var_names, "edges": [e.tolist() for e in edges], "widths": widths,
                     "n_states": n_states, "value_units": value_units, "fps": fps,
                     "state_vars": "physical" if uses_physical(unit) else "pcs",
                     "occupied_joint": int((occ > 0).sum()), "occupied_per_var": occ_var},
                    "04", "bin_edges", unit, sp, set_name)
        print(f"  joint: occupied states {int((occ > 0).sum())} / {n_states}  valid state rows {int((st >= 0).sum())}")
        if DO_JOINT:
            fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
            axes[0].bar(np.arange(n_states), occ, width=1.0, color="#4C78A8")
            axes[0].set_yscale("log")
            axes[0].set_xlabel("joint state id")
            axes[0].set_ylabel("frames")
            ranked = np.sort(occ[occ > 0])[::-1]
            axes[1].loglog(np.arange(1, len(ranked) + 1), ranked, ".", ms=3, color="#4C78A8")
            axes[1].set_xlabel("rank")
            axes[1].set_ylabel("frames")
            fig.suptitle(f"{set_name} {unit_title(unit)}: joint state occupancy, occupied {int((occ > 0).sum())} / {n_states}")
            save_fig(fig, "04", "occupancy_joint", unit, sp, set_name)
        if DO_PER_PC:
            fig, axes = plt.subplots(1, len(var_names), figsize=(4 * len(var_names), 3.4), squeeze=False)
            for ax, name, w in zip(axes[0], var_names, widths):
                b = table[f"bin_{name}"].to_numpy()
                c = np.bincount(b[b >= 0], minlength=w)
                ax.bar(np.arange(w), c, color="#F58518")
                ax.set_yscale("log")
                ax.set_xlabel(f"{name} bin")
                ax.set_title(f"{name}: occupied {int((c > 0).sum())} / {w}", fontsize=9)
                print(f"  per_pc {name}: occupied {int((c > 0).sum())} / {w}")
            axes[0][0].set_ylabel("frames")
            fig.suptitle(f"{set_name} {unit_title(unit)}: per-variable occupancy")
            save_fig(fig, "04", "occupancy_per_pc", unit, sp, set_name)
L.log("04", "done")


[K FM1 (F->M1)] variables ['PC1', 'PC2', 'PC3']  frames in segments 658920
  PC1: values -263.814..168.399  edges -263.814..168.399  outside / NaN 0 (0.000%)
  PC2: values -143.714..167.547  edges -143.714..167.547  outside / NaN 0 (0.000%)
  PC3: values -29.275..28.880  edges -29.275..28.880  outside / NaN 0 (0.000%)


  csv data/04_microstates__FM1__bins10-pcs1_2_3__K.csv  rows 658920
  json data/04_bin_edges__FM1__bins10-pcs1_2_3__K.json
  joint: occupied states 867 / 1000  valid state rows 658920


  fig figures/04_occupancy_joint__FM1__bins10-pcs1_2_3__K.png
  per_pc PC1: occupied 10 / 10
  per_pc PC2: occupied 10 / 10
  per_pc PC3: occupied 10 / 10


  fig figures/04_occupancy_per_pc__FM1__bins10-pcs1_2_3__K.png



[K FM2 (F->M2)] variables ['PC1', 'PC2', 'PC3']  frames in segments 657010
  PC1: values -160.197..247.092  edges -160.197..247.092  outside / NaN 0 (0.000%)
  PC2: values -238.044..184.236  edges -238.044..184.236  outside / NaN 0 (0.000%)
  PC3: values -34.094..31.749  edges -34.094..31.749  outside / NaN 0 (0.000%)


  csv data/04_microstates__FM2__bins10-pcs1_2_3__K.csv  rows 657010
  json data/04_bin_edges__FM2__bins10-pcs1_2_3__K.json
  joint: occupied states 580 / 1000  valid state rows 657010


... (38 outputs left out)


[KP triple (raw coordinates)] variables ['PC1', 'PC2', 'PC3']  frames in segments 1305220
  PC1: values -318.026..352.681  edges -318.026..352.681  outside / NaN 0 (0.000%)
  PC2: values -341.025..283.525  edges -341.025..283.525  outside / NaN 0 (0.000%)
  PC3: values -282.960..334.389  edges -282.960..334.389  outside / NaN 0 (0.000%)


  csv data/04_microstates__triple__bins10-pcs1_2_3__KP.csv  rows 1305220
  json data/04_bin_edges__triple__bins10-pcs1_2_3__KP.json
  joint: occupied states 809 / 1000  valid state rows 1305220


  fig figures/04_occupancy_joint__triple__bins10-pcs1_2_3__KP.png
  per_pc PC1: occupied 10 / 10
  per_pc PC2: occupied 10 / 10
  per_pc PC3: occupied 10 / 10


  fig figures/04_occupancy_per_pc__triple__bins10-pcs1_2_3__KP.png


## Step 5 — 转移概率矩阵

**做什么**：对每个 τ ∈ `TAUS`（帧；图里同时写秒）：所有有效的 (t, t+τ) 配成一对，两帧都有有效状态且在同一连续片段内；所有时刻、所有视频合并进同一个矩阵；稀疏矩阵计数后按行归一化（没有数据的行为 0）。

**输出**：`data/05_transitions_joint.npz`、`data/05_transitions_per_pc.npz`（计数矩阵和概率矩阵）；图 `05_transition_joint`（对数色标，0 为白色）、`05_transition_per_pc`。打印有效配对数、每行之和、对角线占比。

In [8]:
# Step 5: lag-tau transition matrices.
L.clear_log("05")
for set_name in RUN_SETS:
    for unit in RUN_UNITS:
        sp = state_params(unit)
        info = L.load_json("04", "bin_edges", unit, sp, set_name)
        fps = float(info["fps"])
        df = load_micro(unit, set_name)
        segc = micro_seg(df)
        states = df["state"].to_numpy(np.int64)
        for tau in TAUS:
            tp = tau_params(unit, tau)
            head = f"{set_name} {unit_title(unit)}  tau = {tau} frames ({tau / fps:.3f} s)"
            if DO_JOINT:
                counts, n_pairs = S.transition_counts(states, segc, tau, info["n_states"])
                prob = S.row_normalize(counts)
                rs = np.asarray(prob.sum(axis=1)).ravel()
                has = np.asarray(counts.sum(axis=1)).ravel() > 0
                diag = float(counts.diagonal().sum() / max(counts.sum(), 1))
                print(f"[{head}] joint: pairs {n_pairs}  rows with data {int(has.sum())}  "
                      f"max|row sum - 1| {np.abs(rs[has] - 1).max():.1e}  diagonal share {diag:.3f}")
                L.save_npz({**S.sparse_to_arrays("counts", counts), **S.sparse_to_arrays("prob", prob),
                            "n_pairs": n_pairs, "tau": tau}, "05", "transitions_joint", unit, tp, set_name, verbose=False)
                dense = prob.toarray()
                cmap = plt.get_cmap("viridis").copy()
                cmap.set_bad("white")
                fig, ax = plt.subplots(figsize=(8, 7))
                im = ax.imshow(np.ma.masked_equal(dense, 0), cmap=cmap, interpolation="nearest",
                               norm=LogNorm(vmin=dense[dense > 0].min(), vmax=1.0))
                fig.colorbar(im, ax=ax, fraction=0.046, label="P(s_t+tau | s_t)")
                ax.set_xlabel("state at t + tau")
                ax.set_ylabel("state at t")
                ax.set_title(f"{head}\njoint transition matrix, pairs {n_pairs}, diagonal share {diag:.3f}", fontsize=10)
                save_fig(fig, "05", "transition_joint", unit, tp, set_name)
            if DO_PER_PC:
                arrays = {"tau": tau}
                fig, axes = plt.subplots(1, len(info["variables"]), figsize=(4.6 * len(info["variables"]), 4.2), squeeze=False)
                for ax, name, w in zip(axes[0], info["variables"], info["widths"]):
                    b = df[f"bin_{name}"].to_numpy(np.int64)
                    c, n = S.transition_counts(b, segc, tau, int(w))
                    p = S.row_normalize(c).toarray()
                    arrays[f"counts_{name}"] = c.toarray()
                    arrays[f"prob_{name}"] = p
                    arrays[f"n_pairs_{name}"] = n
                    d = float(c.diagonal().sum() / max(c.sum(), 1))
                    im = ax.imshow(p, cmap="viridis", vmin=0, vmax=max(p.max(), 1e-9), interpolation="nearest")
                    for (i, j), v in np.ndenumerate(p):
                        if v >= 0.01:
                            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=5,
                                    color="white" if v < 0.6 * p.max() else "black")
                    ax.set_title(f"{name}: pairs {n}, diagonal {d:.3f}", fontsize=9)
                    ax.set_xlabel("bin at t + tau")
                    ax.set_ylabel("bin at t")
                    print(f"[{head}] per_pc {name}: pairs {n}  diagonal share {d:.3f}")
                fig.suptitle(head + "  per-variable transition matrices", fontsize=10)
                save_fig(fig, "05", "transition_per_pc", unit, tp, set_name)
                L.save_npz(arrays, "05", "transitions_per_pc", unit, tp, set_name, verbose=False)
L.log("05", "done")

[K FM1 (F->M1)  tau = 7 frames (0.042 s)] joint: pairs 651438  rows with data 866  max|row sum - 1| 3.3e-16  diagonal share 0.722


  fig figures/05_transition_joint__FM1__tau7-bins10-pcs1_2_3__K.png
[K FM1 (F->M1)  tau = 7 frames (0.042 s)] per_pc PC1: pairs 651438  diagonal share 0.956
[K FM1 (F->M1)  tau = 7 frames (0.042 s)] per_pc PC2: pairs 651438  diagonal share 0.946
[K FM1 (F->M1)  tau = 7 frames (0.042 s)] per_pc PC3: pairs 651438  diagonal share 0.792


  fig figures/05_transition_per_pc__FM1__tau7-bins10-pcs1_2_3__K.png
[K FM1 (F->M1)  tau = 17 frames (0.102 s)] joint: pairs 644131  rows with data 864  max|row sum - 1| 3.3e-16  diagonal share 0.536


  fig figures/05_transition_joint__FM1__tau17-bins10-pcs1_2_3__K.png
[K FM1 (F->M1)  tau = 17 frames (0.102 s)] per_pc PC1: pairs 644131  diagonal share 0.895
[K FM1 (F->M1)  tau = 17 frames (0.102 s)] per_pc PC2: pairs 644131  diagonal share 0.884
[K FM1 (F->M1)  tau = 17 frames (0.102 s)] per_pc PC3: pairs 644131  diagonal share 0.662


  fig figures/05_transition_per_pc__FM1__tau17-bins10-pcs1_2_3__K.png


[K FM2 (F->M2)  tau = 7 frames (0.042 s)] joint: pairs 649226  rows with data 575  max|row sum - 1| 3.3e-16  diagonal share 0.761


... (50 outputs left out)

  fig figures/05_transition_joint__triple__tau7-bins10-pcs1_2_3__KP.png
[KP triple (raw coordinates)  tau = 7 frames (0.042 s)] per_pc PC1: pairs 1283662  diagonal share 0.966
[KP triple (raw coordinates)  tau = 7 frames (0.042 s)] per_pc PC2: pairs 1283662  diagonal share 0.968
[KP triple (raw coordinates)  tau = 7 frames (0.042 s)] per_pc PC3: pairs 1283662  diagonal share 0.966


  fig figures/05_transition_per_pc__triple__tau7-bins10-pcs1_2_3__KP.png
[KP triple (raw coordinates)  tau = 17 frames (0.102 s)] joint: pairs 1263619  rows with data 807  max|row sum - 1| 3.3e-16  diagonal share 0.792


  fig figures/05_transition_joint__triple__tau17-bins10-pcs1_2_3__KP.png
[KP triple (raw coordinates)  tau = 17 frames (0.102 s)] per_pc PC1: pairs 1263619  diagonal share 0.919
[KP triple (raw coordinates)  tau = 17 frames (0.102 s)] per_pc PC2: pairs 1263619  diagonal share 0.928
[KP triple (raw coordinates)  tau = 17 frames (0.102 s)] per_pc PC3: pairs 1263619  diagonal share 0.922


  fig figures/05_transition_per_pc__triple__tau17-bins10-pcs1_2_3__KP.png


## Step 5b — Infomap 聚类与块对角转移图（必做）

**后端**：先试 python-igraph 的 `community_infomap(trials=N_TRIALS)`，再试独立的 `infomap` 包；都失败则抛出异常并打印安装命令。整个 τ 扫描用同一个后端；后端和版本写入 `params.json` 和图标题。

**做什么**（只对 joint 模式）：去掉从未出现的状态；转移概率矩阵作为加权有向图的边权；`KEEP_SELF_LOOPS` 决定是否保留对角线。簇按覆盖有效帧数降序编号；`CLUSTER_SELECT="top_coverage"` 时画累积覆盖达到 `COVERAGE` 的前若干个簇（其它合并成 "rest"）。

**输出**（每个 τ）：
- (a) `05b_cluster_matrix` / `05b_cluster_matrix_nodiag`：簇×簇转移图（主图），格内数值保留 2 位有效数字，< 0.01 不标；坐标轴写簇编号、每簇状态数和覆盖帧数。
- (b) `05b_state_block` / `05b_state_block_nodiag`：状态级块对角图，行列按"簇 → 簇内状态"排序，0 为白色，对数色标，只在簇边界画细线（12×12 英寸，`DPI_BIG`）。
- (c) `05b_state_block_rare` / `_rare_nodiag`：去掉访问次数 < `MIN_VISITS` 的状态后重画 (b)。
- (d) `05b_cluster_zoom_c<k>`：覆盖帧数最多的 `TOP_ZOOM` 个簇的簇内状态×状态子矩阵。
- (e) 可选的 plotly 交互热图（`MAKE_HTML`）。
- 定量检查 `data/05b_cluster_stats.csv`（每簇簇内转移占比、去掉对角线后的占比、状态数、覆盖帧数、平均停留时间、各变量取值范围）、`data/05b_within_summary.csv`（整体加权平均和打乱 `N_SHUFFLE` 次的均值与范围）。**按簇排序之后图总会显得有块状，请对照打乱参照来看。**
- τ 扫描（仿 Fig. S11A）：`05b_tau_scan` 图（主图：最大簇的状态数 / 状态空间格子总数；插图：簇数），`data/05b_tau_scan.csv`。
- `data/05b_cluster_labels.csv`：每个有效帧的簇编号（−1 = 未分配），带视频名和帧号；`data/05b_state_clusters.csv`：状态 → 簇。

In [9]:
# Step 5b: Infomap on the joint transition graph, block-diagonal views, checks, tau scan.
L.clear_log("05b")
if not DO_JOINT:
    raise RuntimeError("Step 5b needs the joint states: set DO_JOINT = True and rerun Steps 4 and 5")
INFOMAP = S.resolve_infomap_backend()
BACKEND_TAG = f"{INFOMAP['backend']} {INFOMAP['version']}"
print("Infomap backend:", BACKEND_TAG, "|", INFOMAP["seed_note"])
L.update_params("infomap", {**INFOMAP, "N_TRIALS": N_TRIALS, "N_TRIALS_SCAN": N_TRIALS_SCAN,
                            "seed": INFOMAP_SEED, "KEEP_SELF_LOOPS": KEEP_SELF_LOOPS, "workers": INFOMAP_WORKERS})
CMAP_W = plt.get_cmap("viridis").copy()
CMAP_W.set_bad("white")


def graph_input(counts):
    c = sparse.csr_matrix(counts, dtype=np.float64)
    keep = np.flatnonzero((np.asarray(c.sum(axis=1)).ravel() > 0) | (np.asarray(c.sum(axis=0)).ravel() > 0))
    if len(keep) == 0:
        raise RuntimeError("the joint transition matrix has no occupied state")
    sub = c[keep][:, keep].tolil()
    if not KEEP_SELF_LOOPS:
        sub.setdiag(0)
    sub = sub.tocsr()
    sub.eliminate_zeros()
    return keep, S.row_normalize(sub)


def run_jobs(jobs, trials):
    # Independent Infomap runs (same backend, same seed) in parallel processes.
    from concurrent.futures import ProcessPoolExecutor
    if INFOMAP_WORKERS <= 1:
        return {k: S.run_infomap(p, trials, INFOMAP_SEED, INFOMAP["backend"]) for k, p in jobs}
    with ProcessPoolExecutor(max_workers=INFOMAP_WORKERS) as ex:
        futures = {k: ex.submit(S.run_infomap, p, trials, INFOMAP_SEED, INFOMAP["backend"]) for k, p in jobs}
        return {k: f.result() for k, f in futures.items()}


def relabel(raw, keep, n_states, state_frames):
    # Renumber modules by covered frames (0 = most); states outside the graph get -1.
    lab = np.full(n_states, -1, dtype=np.int64)
    lab[keep] = raw
    k = int(raw.max()) + 1
    cover = np.bincount(lab[keep], weights=state_frames[keep], minlength=k)
    order = np.lexsort((np.arange(k), -cover))
    new = np.empty(k, dtype=np.int64)
    new[order] = np.arange(k)
    out = np.full(n_states, -1, dtype=np.int64)
    out[keep] = new[raw]
    return out, cover[order]


def select_clusters(cover, n_valid):
    if CLUSTER_SELECT == "all":
        return np.arange(len(cover))
    if CLUSTER_SELECT != "top_coverage":
        raise ValueError("CLUSTER_SELECT must be 'top_coverage' or 'all'")
    cum = np.cumsum(cover) / max(n_valid, 1)
    n = int(np.searchsorted(cum, COVERAGE) + 1)
    return np.arange(min(n, len(cover)))


def cluster_matrix(counts, lab, sel):
    k = int(lab.max()) + 1
    rest = len(sel) < k
    disp = np.full(k, len(sel), dtype=np.int64)
    disp[sel] = np.arange(len(sel))
    n = len(sel) + int(rest)
    coo = counts.tocoo()
    ok = (lab[coo.row] >= 0) & (lab[coo.col] >= 0)
    C = np.zeros((n, n))
    np.add.at(C, (disp[lab[coo.row[ok]]], disp[lab[coo.col[ok]]]), coo.data[ok])
    rs = C.sum(axis=1, keepdims=True)
    return C, np.divide(C, rs, out=np.zeros_like(C), where=rs > 0), rest


def draw_cluster_matrix(P, labels, title, mask_diag):
    n = len(P)
    show = np.ma.array(P, mask=np.eye(n, dtype=bool) if mask_diag else np.zeros_like(P, dtype=bool))
    vmax = float(show.max()) if show.count() else 1.0
    side = min(22.0, max(6.0, 0.55 * n + 3))
    fig, ax = plt.subplots(figsize=(side, side * 0.9))
    cmap = plt.get_cmap("magma_r").copy()
    cmap.set_bad("white")
    im = ax.imshow(show, cmap=cmap, vmin=0, vmax=vmax if vmax > 0 else 1, interpolation="nearest")
    fs = max(4, min(9, int(140 / max(n, 1))))
    for (i, j), v in np.ndenumerate(P):
        if mask_diag and i == j:
            continue
        if v >= 0.01:
            ax.text(j, i, f"{v:.2g}", ha="center", va="center", fontsize=fs,
                    color="white" if v > 0.55 * vmax else "black")
    ax.set_xticks(np.arange(n), labels, rotation=90, fontsize=fs)
    ax.set_yticks(np.arange(n), labels, fontsize=fs)
    ax.set_xlabel("cluster at t + tau")
    ax.set_ylabel("cluster at t")
    fig.colorbar(im, ax=ax, fraction=0.04, label="P(cluster_t+tau | cluster_t)")
    ax.set_title(title, fontsize=10)
    return fig


def draw_block(sub, bounds, names, title, mask_diag):
    mask = sub <= 0
    if mask_diag:
        mask |= np.eye(len(sub), dtype=bool)
    shown = sub[~mask]
    fig, ax = plt.subplots(figsize=(12, 12))
    if shown.size:
        im = ax.imshow(np.ma.array(sub, mask=mask), cmap=CMAP_W, interpolation="nearest",
                       norm=LogNorm(vmin=float(shown.min()), vmax=float(shown.max())))
        fig.colorbar(im, ax=ax, fraction=0.04, label="P(s_t+tau | s_t), log scale")
    for b in bounds[1:-1]:
        ax.axhline(b - 0.5, color="0.25", lw=0.4)
        ax.axvline(b - 0.5, color="0.25", lw=0.4)
    centers = [(a + b) / 2 - 0.5 for a, b in zip(bounds[:-1], bounds[1:])]
    fs = max(3, min(9, int(300 / max(len(names), 1))))
    ax.set_xticks(centers, names, rotation=90, fontsize=fs)
    ax.set_yticks(centers, names, fontsize=fs)
    ax.grid(False)
    ax.set_title(title, fontsize=10)
    return fig


def block_order(lab, clusters, keep_state=None):
    order, bounds, names = [], [0], []
    for c in clusters:
        st = np.flatnonzero(lab == c)
        if keep_state is not None:
            st = st[keep_state[st]]
        if len(st) == 0:
            continue
        order.extend(st.tolist())
        bounds.append(len(order))
        names.append(f"C{c}")
    return np.asarray(order, dtype=np.int64), bounds, names


def state_label(s, widths):
    return ",".join(str(v) for v in S.decode_state(np.array([s]), widths)[0])


# ---- main Infomap runs (all sets, units, taus)
jobs, CTX = [], {}
for set_name in RUN_SETS:
    for unit in RUN_UNITS:
        info = L.load_json("04", "bin_edges", unit, state_params(unit), set_name)
        for tau in TAUS:
            z = L.load_npz("05", "transitions_joint", unit, tau_params(unit, tau), set_name)
            counts = S.arrays_to_sparse("counts", z)
            keep, prob = graph_input(counts)
            CTX[(set_name, unit, tau)] = {"counts": counts, "keep": keep, "info": info, "n_pairs": int(z["n_pairs"])}
            jobs.append(((set_name, unit, tau), prob))
t0 = time.time()
RESULTS = run_jobs(jobs, N_TRIALS)
print(f"{len(jobs)} Infomap runs x {N_TRIALS} trials: {time.time() - t0:.0f} s")

for set_name in RUN_SETS:
    for unit in RUN_UNITS:
        sp = state_params(unit)
        df = load_micro(unit, set_name)
        orig = df["copy"].to_numpy() == 0
        st_orig = df["state"].to_numpy(np.int64)[orig]
        seg_orig = micro_seg(df)[orig]
        for tau in TAUS:
            ctx = CTX[(set_name, unit, tau)]
            info, counts, keep = ctx["info"], ctx["counts"], ctx["keep"]
            n_states, widths, fps = info["n_states"], info["widths"], float(info["fps"])
            tp = tau_params(unit, tau)
            head = f"{set_name} {unit_title(unit)}  tau = {tau} fr ({tau / fps:.3f} s)"
            raw, codelength = RESULTS[(set_name, unit, tau)]
            state_frames = np.bincount(st_orig[st_orig >= 0], minlength=n_states).astype(float)
            lab, cover = relabel(raw, keep, n_states, state_frames)
            n_valid = int((st_orig >= 0).sum())
            k = len(cover)
            sel = select_clusters(cover, n_valid)
            n_st = np.bincount(lab[lab >= 0], minlength=k)
            print(f"\n[{head}] clusters {k}  selected {len(sel)} covering {100 * cover[sel].sum() / n_valid:.2f}% "
                  f"of {n_valid} valid frames  codelength {codelength:.4f}")

            # (a) cluster x cluster
            C, Pc, rest = cluster_matrix(counts, lab, sel)
            labels = [f"C{c} ({n_st[c]} st, {int(cover[c])} fr)" for c in sel]
            if rest:
                rest_ids = np.setdiff1d(np.arange(k), sel)
                labels.append(f"rest ({len(rest_ids)} cl, {int(n_st[rest_ids].sum())} st, {int(cover[rest_ids].sum())} fr)")
            for mask_diag, nm in ((False, "cluster_matrix"), (True, "cluster_matrix_nodiag")):
                note = "diagonal masked (values unchanged)" if mask_diag else "with diagonal"
                fig = draw_cluster_matrix(Pc, labels, f"{head}\ncluster-to-cluster transitions, {note}; {BACKEND_TAG}", mask_diag)
                save_fig(fig, "05b", nm, unit, tp, set_name)

            # (b) state-level block diagonal
            prob_full = S.row_normalize(counts)
            order, bounds, names = block_order(lab, sel)
            sub = prob_full[order][:, order].toarray()
            for mask_diag, nm in ((False, "state_block"), (True, "state_block_nodiag")):
                note = "state diagonal masked" if mask_diag else "with diagonal"
                fig = draw_block(sub, bounds, names, f"{head}\nstates ordered by cluster ({len(order)} states in "
                                 f"{len(names)} selected clusters), {note}; zero = white; {BACKEND_TAG}", mask_diag)
                save_fig(fig, "05b", nm, unit, tp, set_name, dpi=DPI_BIG)

            # (c) rare states removed
            if FILTER_RARE:
                frequent = state_frames >= MIN_VISITS
                order_r, bounds_r, names_r = block_order(lab, sel, frequent)
                removed = np.setdiff1d(order, order_r)
                print(f"  rare filter (< {MIN_VISITS} visits): removed {len(removed)} of {len(order)} states, "
                      f"{100 * state_frames[removed].sum() / n_valid:.3f}% of valid frames")
                sub_r = prob_full[order_r][:, order_r].toarray()
                for mask_diag, nm in ((False, "state_block_rare"), (True, "state_block_rare_nodiag")):
                    note = "state diagonal masked" if mask_diag else "with diagonal"
                    fig = draw_block(sub_r, bounds_r, names_r, f"{head}\nstates with >= {MIN_VISITS} visits "
                                     f"({len(removed)} removed), {note}; {BACKEND_TAG}", mask_diag)
                    save_fig(fig, "05b", nm, unit, tp, set_name, dpi=DPI_BIG)

            # (d) zoom into the largest clusters
            for c in range(min(TOP_ZOOM, k)):
                st = np.flatnonzero(lab == c)
                zsub = prob_full[st][:, st].toarray()
                fig, ax = plt.subplots(figsize=(min(16, 4 + 0.16 * len(st)),) * 2)
                if (zsub > 0).any():
                    im = ax.imshow(np.ma.masked_less_equal(zsub, 0), cmap=CMAP_W, interpolation="nearest",
                                   norm=LogNorm(vmin=zsub[zsub > 0].min(), vmax=zsub.max()))
                    fig.colorbar(im, ax=ax, fraction=0.04, label="P(s_t+tau | s_t)")
                step = max(1, len(st) // 60)
                ticks = np.arange(0, len(st), step)
                tl = [f"{st[i]} ({state_label(st[i], widths)})" for i in ticks]
                ax.set_xticks(ticks, tl, rotation=90, fontsize=5)
                ax.set_yticks(ticks, tl, fontsize=5)
                ax.set_title(f"{head}\ncluster C{c}: {len(st)} states, {int(cover[c])} frames; "
                             f"labels = state (bins {','.join(info['variables'])})", fontsize=9)
                save_fig(fig, "05b", f"cluster_zoom_c{c}", unit, tp, set_name)

            # (e) optional interactive heatmap
            if MAKE_HTML:
                try:
                    import plotly.graph_objects as go
                except ImportError as exc:
                    raise ImportError("MAKE_HTML needs plotly: .venv\\Scripts\\python.exe -m pip install plotly") from exc
                cnt_sub = counts[order][:, order].toarray()
                hover = [[f"from {order[i]} (C{lab[order[i]]}) to {order[j]} (C{lab[order[j]]})<br>"
                          f"P={sub[i, j]:.3g} count={int(cnt_sub[i, j])}" for j in range(len(order))]
                         for i in range(len(order))]
                figp = go.Figure(go.Heatmap(z=np.where(sub > 0, np.log10(np.where(sub > 0, sub, 1)), np.nan),
                                            text=hover, hoverinfo="text", colorbar={"title": "log10 P"}))
                figp.update_layout(title=head, yaxis={"autorange": "reversed"}, width=1000, height=1000)
                hp = L.path("05b", "state_block_interactive", unit, tp, set_name, "html", folder="figures")
                figp.write_html(hp, include_plotlyjs=True)
                print("  html", hp.name)

            # 4. quantitative checks
            w_d = S.per_cluster_within(counts, lab)
            w_n = S.per_cluster_within(counts, lab, drop_diagonal=True)
            overall = S.within_fraction(counts, lab)
            overall_nd = S.within_fraction(counts, lab, drop_diagonal=True)
            rng = np.random.default_rng(INFOMAP_SEED)
            sh, sh_nd = [], []
            for _ in range(N_SHUFFLE):
                perm = lab.copy()
                perm[keep] = rng.permutation(lab[keep])
                sh.append(S.within_fraction(counts, perm))
                sh_nd.append(S.within_fraction(counts, perm, drop_diagonal=True))
            lab_frames = np.where(st_orig >= 0, lab[np.clip(st_orig, 0, None)], -1)
            dw = S.dwell_runs(lab_frames, seg_orig)
            dwell = dw.groupby("cluster")["length"].agg(["mean", "count"]).reindex(np.arange(k))
            decoded = S.decode_state(np.arange(n_states), widths)
            stats = pd.DataFrame({"cluster": np.arange(k), "n_states": n_st, "n_frames": cover.astype(int),
                                  "frame_fraction": cover / n_valid, "selected": np.isin(np.arange(k), sel),
                                  "within_fraction": w_d["within_fraction"].to_numpy(),
                                  "within_fraction_nodiag": w_n["within_fraction"].to_numpy(),
                                  "mean_dwell_frames": dwell["mean"].to_numpy(),
                                  "mean_dwell_s": dwell["mean"].to_numpy() / fps,
                                  "n_dwell_runs": dwell["count"].fillna(0).astype(int).to_numpy()})
            for j, (vname, e) in enumerate(zip(info["variables"], info["edges"])):
                centers = 0.5 * (np.asarray(e)[:-1] + np.asarray(e)[1:])
                lo = [centers[decoded[lab == c, j]].min() for c in range(k)]
                hi = [centers[decoded[lab == c, j]].max() for c in range(k)]
                stats[f"{vname}_min"] = lo
                stats[f"{vname}_max"] = hi
            L.save_csv(stats, "05b", "cluster_stats", unit, tp, set_name, verbose=False)
            summary = {"tau": tau, "tau_s": tau / fps, "n_clusters": k, "n_selected": len(sel),
                       "selected_frame_fraction": cover[sel].sum() / n_valid, "occupied_states": len(keep),
                       "grid_states": n_states, "valid_pairs": ctx["n_pairs"],
                       "largest_cluster_state_fraction": n_st.max() / n_states,
                       "largest_cluster_frame_fraction": cover[0] / n_valid,
                       "within_fraction": overall, "within_fraction_nodiag": overall_nd,
                       "shuffle_mean": np.mean(sh), "shuffle_min": np.min(sh), "shuffle_max": np.max(sh),
                       "shuffle_nodiag_mean": np.mean(sh_nd), "shuffle_nodiag_min": np.min(sh_nd),
                       "shuffle_nodiag_max": np.max(sh_nd), "codelength": codelength, "backend": BACKEND_TAG}
            L.save_csv(pd.DataFrame([summary]), "05b", "within_summary", unit, tp, set_name, verbose=False)
            print(f"  within-cluster share {overall:.3f} (no diagonal {overall_nd:.3f}); shuffled x{N_SHUFFLE}: "
                  f"{np.mean(sh):.3f} [{np.min(sh):.3f}, {np.max(sh):.3f}] (no diagonal {np.mean(sh_nd):.3f} "
                  f"[{np.min(sh_nd):.3f}, {np.max(sh_nd):.3f}])")
            print("  largest clusters:\n" + stats.head(min(8, k))[["cluster", "n_states", "n_frames", "frame_fraction",
                  "within_fraction", "within_fraction_nodiag", "mean_dwell_s"]].round(3).to_string(index=False))

            # 6. per-frame cluster labels and the state table
            o = df[orig]
            L.save_csv(pd.DataFrame({"video": o["video"].to_numpy(), "frame": o["frame"].to_numpy(),
                                     "state": st_orig, "cluster": lab_frames}),
                       "05b", "cluster_labels", unit, tp, set_name, verbose=False)
            stab = pd.DataFrame({"state": np.arange(n_states), "cluster": lab, "n_frames": state_frames.astype(int)})
            for j, vname in enumerate(info["variables"]):
                stab[f"bin_{vname}"] = decoded[:, j]
            L.save_csv(stab, "05b", "state_clusters", unit, tp, set_name, verbose=False)
            print(f"  frames without a cluster (-1): {int((lab_frames < 0).sum())}")

# ---- 5. tau scan
scan_jobs, SCAN = [], {}
for set_name in RUN_SETS:
    for unit in RUN_UNITS:
        info = L.load_json("04", "bin_edges", unit, state_params(unit), set_name)
        df = load_micro(unit, set_name)
        segc = micro_seg(df)
        states = df["state"].to_numpy(np.int64)
        for tau in TAU_SCAN:
            counts, n_pairs = S.transition_counts(states, segc, int(tau), info["n_states"])
            keep, prob = graph_input(counts)
            SCAN[(set_name, unit, tau)] = (counts, keep, n_pairs, info)
            scan_jobs.append(((set_name, unit, tau), prob))
t0 = time.time()
SCAN_RES = run_jobs(scan_jobs, N_TRIALS_SCAN)
print(f"\ntau scan: {len(scan_jobs)} Infomap runs x {N_TRIALS_SCAN} trials: {time.time() - t0:.0f} s")
for set_name in RUN_SETS:
    for unit in RUN_UNITS:
        sp = state_params(unit)
        rows = []
        for tau in TAU_SCAN:
            counts, keep, n_pairs, info = SCAN[(set_name, unit, tau)]
            raw, _cl = SCAN_RES[(set_name, unit, tau)]
            lab = np.full(info["n_states"], -1, dtype=np.int64)
            lab[keep] = raw
            sizes = np.bincount(raw)
            rows.append({"tau": tau, "tau_s": tau / float(info["fps"]), "n_clusters": len(sizes),
                         "largest_cluster_state_fraction": sizes.max() / info["n_states"],
                         "occupied_states": len(keep), "valid_pairs": n_pairs,
                         "within_fraction": S.within_fraction(counts, lab)})
        scan = pd.DataFrame(rows)
        L.save_csv(scan, "05b", "tau_scan", unit, sp, set_name, verbose=False)
        print(f"[{set_name} {unit_title(unit)}] tau scan\n" + scan.round(4).to_string(index=False))
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.plot(scan["tau"], scan["largest_cluster_state_fraction"], "o-", color="#4C78A8")
        for t in TAUS:
            ax.axvline(t, color="#E45756", ls="--", lw=1)
        ax.set_xlabel("tau (frames)")
        ax.set_ylabel("largest cluster states / all grid states")
        sec = ax.secondary_xaxis("top", functions=(lambda f: f / float(info["fps"]), lambda s: s * float(info["fps"])))
        sec.set_xlabel("tau (s)")
        ins = ax.inset_axes([0.58, 0.5, 0.38, 0.38])
        ins.plot(scan["tau"], scan["n_clusters"], "s-", ms=3, color="0.3")
        ins.set_title("number of clusters", fontsize=8)
        ins.tick_params(labelsize=7)
        ax.set_title(f"{set_name} {unit_title(unit)}: tau scan ({N_TRIALS_SCAN} trials, {BACKEND_TAG}); "
                     f"dashed = TAUS", fontsize=9)
        save_fig(fig, "05b", "tau_scan", unit, sp, set_name)
L.log("05b", "done")
if not MAKE_HTML:
    L.log("05b", "skipped", "interactive plotly heatmaps (MAKE_HTML = False)")

Infomap backend: python-igraph 1.0.0 | community_infomap has no seed argument; igraph uses Python's random module, which is seeded before every call


24 Infomap runs x 500 trials: 30 s



[K FM1 (F->M1)  tau = 7 fr (0.042 s)] clusters 305  selected 207 covering 95.04% of 658920 valid frames  codelength 4.0234


  fig figures/05b_cluster_matrix__FM1__tau7-bins10-pcs1_2_3__K.png


  fig figures/05b_cluster_matrix_nodiag__FM1__tau7-bins10-pcs1_2_3__K.png


  fig figures/05b_state_block__FM1__tau7-bins10-pcs1_2_3__K.png


... (277 outputs left out)

  fig figures/05b_tau_scan__M1M2__bins10-pcs1_2_3__P.png
[P triple (raw coordinates)] tau scan
 tau  tau_s  n_clusters  largest_cluster_state_fraction  occupied_states  valid_pairs  within_fraction
   1 0.0060         580                           0.002              597       648334           0.9869
   2 0.0120         568                           0.003              597       646471           0.9759
   3 0.0181         547                           0.003              597       644851           0.9653
   5 0.0301         512                           0.004              597       642018           0.9449
   7 0.0422         484                           0.005              597       639507           0.9258
  10 0.0602         438                           0.005              597       636092           0.9009
  12 0.0723         397                           0.006              597       633980           0.8873
  15 0.0904         344                           0.006              596       63

  fig figures/05b_tau_scan__FM1__bins10-pcs1_2_3__KP.png
[KP FM2 (F->M2)] tau scan
 tau  tau_s  n_clusters  largest_cluster_state_fraction  occupied_states  valid_pairs  within_fraction
   1 0.0060         365                           0.004              401      1309277           0.9640
   2 0.0120         336                           0.003              401      1306586           0.9407
   3 0.0181         314                           0.004              401      1304221           0.9208
   5 0.0301         255                           0.005              401      1300005           0.8983
   7 0.0422         219                           0.006              401      1296202           0.8833
  10 0.0602         192                           0.007              401      1291001           0.8591
  12 0.0723         176                           0.006              401      1287760           0.8499
  15 0.0904         162                           0.007              401      1283100        

  fig figures/05b_tau_scan__M1M2__bins10-pcs1_2_3__KP.png
[KP triple (raw coordinates)] tau scan
 tau  tau_s  n_clusters  largest_cluster_state_fraction  occupied_states  valid_pairs  within_fraction
   1 0.0060         783                           0.003              808      1300630           0.9842
   2 0.0120         765                           0.003              808      1297064           0.9702
   3 0.0181         743                           0.004              808      1293966           0.9567
   5 0.0301         708                           0.005              808      1288517           0.9310
   7 0.0422         664                           0.005              808      1283662           0.9076
  10 0.0602         577                           0.007              808      1277099           0.8773
  12 0.0723         517                           0.007              808      1273040           0.8623
  15 0.0904         437                           0.007              808      1

  fig figures/05b_tau_scan__triple__bins10-pcs1_2_3__KP.png


## Step 6 — K 与 P 的一致性检查（每个单元一次，只输出数值和图）

**做什么**：
- (a) K、P 各自前 `N_CMP` 个 PC 的载荷两两 |余弦相似度|（热图 + csv）；两组方差解释比例对比图；每个 K 的 PC 在 P 中最相似的 PC 及数值（K、P 用同一套特征顺序）。
- (b) 按 (a) 的配对，把对应 PC 的 ACF 画在同一张图，输出 1/e 相关时间对比表。
- (c) 跨组应用（K→P、P→K，不重新拟合）：用一组的 PCA 均值和载荷投影另一组，用一组的分格边界离散化另一组；输出每个变量落在分格范围之外的帧占比、两个状态占用分布的并排图、Jensen-Shannon 距离、τ ∈ `TAUS` 转移矩阵的对角线概率对比。
- (d) 参照基线（`DO_DAY_REFERENCE`）：同一组内按天拆成两半，重复 (a) 的载荷相似度和 (c) 的 JS 距离，和 K–P 放在同一张表。同一组不同天不是独立样本。
- (e) 写 `data/06_consistency.csv`；Step 8 把它并入 `summary.csv`。不设阈值，不写结论。

需要 `RUN_SETS` 同时包含 K 和 P。

In [10]:
# Step 6: K vs P consistency per unit (numbers and figures only).
L.clear_log("06")


def comp_cos(ca, cb, n):
    n = min(n, ca.shape[1], cb.shape[1])
    return np.abs(ca[:, :n].T @ cb[:, :n])


def occupancy(st, n):
    return np.bincount(st[st >= 0], minlength=n).astype(float)


def pc_edges(scores, mask):
    pct = tuple(PERCENTILE) if BIN_BY_PERCENTILE else None
    edges = [S.equal_edges(scores[mask, p - 1], N_BINS, pct) for p in PCS_FOR_STATE]
    return {"variables": [f"PC{p}" for p in PCS_FOR_STATE], "edges": [e.tolist() for e in edges],
            "widths": [N_BINS] * len(edges), "n_states": N_BINS ** len(edges)}


def day_reference(unit, group):
    data = load_set(group)
    seg = load_seg(unit, group)
    use = seg >= 0
    days = sorted(set(int(d) for d in data["day_v"]))
    h = len(days) // 2
    if h == 0:
        return None
    fday = data["day_v"][data["video"]]
    halves = []
    for dsel in (days[:h], days[h:]):
        m = use & np.isin(fday, dsel)
        seg_h = np.where(m, seg, -1)
        feat, _ = unit_features(data["nodes"], unit, m)
        model = S.fit_pca(feat[m], PCA_STANDARDIZE, FIX_PC_SIGN)
        if uses_physical(unit):
            info = L.load_json("04", "bin_edges", unit, state_params(unit), group)
        else:
            info = pc_edges(S.pca_scores(feat, model), m)
        halves.append((dsel, seg_h, model, info))
    (dA, sA, mA, iA), (dB, sB, mB, iB) = halves
    cos = comp_cos(mA["components"], mB["components"], N_CMP)
    best = cos.max(axis=1)
    own_a, _ = states_in_space(unit, data, sA, mA, iA)
    b_in_a, out_ab = states_in_space(unit, data, sB, mA, iA)
    own_b, _ = states_in_space(unit, data, sB, mB, iB)
    a_in_b, out_ba = states_in_space(unit, data, sA, mB, iB)
    n = iA["n_states"]
    return {"comparison": f"{group} day {dA} vs day {dB} (same fish, not independent)",
            "a": f"{group} day {dA}", "b": f"{group} day {dB}",
            "mean_best_abs_cos": float(best.mean()), "best_abs_cos": " ".join(f"{v:.3f}" for v in best),
            "js_a_to_b": S.js_distance(occupancy(own_a, n), occupancy(b_in_a, n)),
            "js_b_to_a": S.js_distance(occupancy(own_b, n), occupancy(a_in_b, n)),
            "outside_a_to_b": json.dumps({k: round(v, 5) for k, v in out_ab.items()}),
            "outside_b_to_a": json.dumps({k: round(v, 5) for k, v in out_ba.items()})}


def run_step6(unit):
    sp, p1 = state_params(unit), unit_params(unit)
    print(f"\n===== {unit_title(unit)}")
    mK, mP = load_model(unit, "K"), load_model(unit, "P")
    if list(mK["feature_names"]) != list(mP["feature_names"]):
        raise RuntimeError(f"{unit}: K and P use different feature orders")
    # (a) loading similarity and variance
    cos = comp_cos(mK["components"], mP["components"], N_CMP)
    n = cos.shape[0]
    match = cos.argmax(axis=1)
    best = cos[np.arange(n), match]
    L.save_csv(pd.DataFrame(cos, columns=[f"P_PC{j + 1}" for j in range(n)]).assign(K_pc=np.arange(1, n + 1))
               [["K_pc"] + [f"P_PC{j + 1}" for j in range(n)]], "06", "loading_cos", unit, p1, "KvsP", verbose=False)
    mt = pd.DataFrame({"K_pc": np.arange(1, n + 1), "best_P_pc": match + 1, "abs_cos": best})
    L.save_csv(mt, "06", "loading_best_match", unit, p1, "KvsP", verbose=False)
    print("  (a) best P match of each K PC:\n" + mt.round(3).to_string(index=False))
    fig, ax = plt.subplots(figsize=(1.0 * n + 2.5, 0.9 * n + 1.5))
    im = ax.imshow(cos, cmap="viridis", vmin=0, vmax=1)
    for (i, j), v in np.ndenumerate(cos):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v < 0.6 else "black")
    ax.set_xticks(np.arange(n), [f"P PC{j + 1}" for j in range(n)])
    ax.set_yticks(np.arange(n), [f"K PC{i + 1}" for i in range(n)])
    fig.colorbar(im, ax=ax, fraction=0.046, label="|cosine similarity|")
    ax.set_title(f"{unit_title(unit)}: |cos| of K and P loadings", fontsize=10)
    save_fig(fig, "06", "loading_cos", unit, p1, "KvsP")
    n_all = len(mK["explained"])
    fig, ax = plt.subplots(figsize=(max(6, 0.4 * n_all + 2), 3.8))
    xs = np.arange(1, n_all + 1)
    ax.bar(xs - 0.2, 100 * mK["explained"], width=0.4, label="K", color="#4C78A8")
    ax.bar(xs + 0.2, 100 * mP["explained"], width=0.4, label="P", color="#F58518")
    ax.set_yscale("log")
    ax.set_xticks(xs)
    ax.tick_params(axis="x", labelsize=7)
    ax.set_xlabel("PC")
    ax.set_ylabel("explained variance (%)")
    ax.legend()
    ax.set_title(f"{unit_title(unit)}: explained variance per PC, K vs P", fontsize=10)
    save_fig(fig, "06", "variance_compare", unit, p1, "KvsP")
    # (b) ACF of matched PCs
    aK, aP = L.load_npz("03", "acf", unit, p1, "K"), L.load_npz("03", "acf", unit, p1, "P")
    tK, tP = L.load_csv("03", "acf_times", unit, p1, "K"), L.load_csv("03", "acf_times", unit, p1, "P")
    cols = 3
    rows_n = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4.2 * cols, 3.0 * rows_n), squeeze=False)
    acf_rows = []
    for i in range(n):
        ax = axes[i // cols][i % cols]
        j = int(match[i])
        ax.plot(aK["lag_s"], aK["curve"][:, i], color="#4C78A8", label=f"K PC{i + 1}")
        ax.plot(aP["lag_s"], aP["curve"][:, j], color="#F58518", label=f"P PC{j + 1}")
        ax.axhline(np.exp(-1), color="k", ls="--", lw=0.7)
        ax.set_title(f"K PC{i + 1} vs P PC{j + 1} (|cos| {best[i]:.2f})", fontsize=9)
        ax.set_xlabel("lag (s)")
        ax.legend(fontsize=7)
        acf_rows.append({"K_pc": i + 1, "P_pc": j + 1, "abs_cos": best[i],
                         "tau_1e_K_s": tK.loc[i, "tau_1e_s"], "tau_1e_P_s": tP.loc[j, "tau_1e_s"]})
    for k in range(n, rows_n * cols):
        axes[k // cols][k % cols].axis("off")
    fig.suptitle(f"{unit_title(unit)}: ACF of matched PCs (dashed = 1/e)", fontsize=10)
    save_fig(fig, "06", "acf_compare", unit, p1, "KvsP")
    acf_tab = pd.DataFrame(acf_rows)
    L.save_csv(acf_tab, "06", "acf_compare", unit, p1, "KvsP", verbose=False)
    print("  (b) 1/e times of matched PCs (s):\n" + acf_tab.round(3).to_string(index=False))
    # (c) cross-application without refitting
    data = {g: load_set(g) for g in ("K", "P")}
    segs = {g: load_seg(unit, g) for g in ("K", "P")}
    out_row = {"comparison": "K vs P", "a": "K", "b": "P", "mean_best_abs_cos": float(best.mean()),
               "best_abs_cos": " ".join(f"{v:.3f}" for v in best)}
    for src, tgt in (("K", "P"), ("P", "K")):
        info = L.load_json("04", "bin_edges", unit, sp, src)
        model = load_model(unit, src)
        own, _ = states_in_space(unit, data[src], segs[src], model, info)
        cross, outside = states_in_space(unit, data[tgt], segs[tgt], model, info)
        ns = info["n_states"]
        oc_own, oc_x = occupancy(own, ns), occupancy(cross, ns)
        js = S.js_distance(oc_own, oc_x)
        tag = "a_to_b" if src == "K" else "b_to_a"
        out_row[f"js_{tag}"] = js
        out_row[f"outside_{tag}"] = json.dumps({k: round(v, 5) for k, v in outside.items()})
        print(f"  (c) {tgt} in the {src} state space: outside {outside}  JS distance {js:.4f}")
        order = np.argsort(-oc_own, kind="stable")
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
        for ax, occ_v, lab_v, c in ((axes[0], oc_own, f"{src} (own)", "#4C78A8"), (axes[1], oc_x, f"{tgt} projected", "#F58518")):
            ax.bar(np.arange(ns), occ_v[order] / max(occ_v.sum(), 1), width=1.0, color=c)
            ax.set_yscale("log")
            ax.set_xlabel(f"state, ordered by {src} occupancy")
            ax.set_title(lab_v, fontsize=9)
        axes[0].set_ylabel("fraction of frames")
        fig.suptitle(f"{unit_title(unit)}: occupancy in the {src} state space ({src} PCA + {src} bin edges), "
                     f"JS distance {js:.3f}", fontsize=10)
        save_fig(fig, "06", f"occupancy_{src}space", unit, sp, "KvsP")
        fig, axes = plt.subplots(1, len(TAUS), figsize=(4.6 * len(TAUS), 4.2), squeeze=False)
        for ax, tau in zip(axes[0], TAUS):
            ca, _ = S.transition_counts(own, segs[src], tau, ns)
            cb, _ = S.transition_counts(cross, segs[tgt], tau, ns)
            pa, pb = S.row_normalize(ca).diagonal(), S.row_normalize(cb).diagonal()
            both = (np.asarray(ca.sum(axis=1)).ravel() > 0) & (np.asarray(cb.sum(axis=1)).ravel() > 0)
            da = float(ca.diagonal().sum() / max(ca.sum(), 1))
            db = float(cb.diagonal().sum() / max(cb.sum(), 1))
            out_row[f"diag_share_{src}space_{src}_tau{tau}"] = da
            out_row[f"diag_share_{src}space_{tgt}_tau{tau}"] = db
            ax.scatter(pa[both], pb[both], s=5, alpha=0.5)
            ax.plot([0, 1], [0, 1], color="k", lw=0.7)
            ax.set_xlabel(f"P_ii, {src} own")
            ax.set_ylabel(f"P_ii, {tgt} projected")
            ax.set_title(f"tau {tau}: diagonal share {da:.3f} vs {db:.3f}; {int(both.sum())} states", fontsize=9)
            print(f"      tau {tau}: diagonal share {src} {da:.4f}  {tgt}-projected {db:.4f}  states in both {int(both.sum())}")
        fig.suptitle(f"{unit_title(unit)}: diagonal transition probabilities in the {src} state space", fontsize=10)
        save_fig(fig, "06", f"diag_compare_{src}space", unit, sp, "KvsP")
    rows = [out_row]
    # (d) within-group day reference
    if DO_DAY_REFERENCE:
        for g in ("K", "P"):
            r = day_reference(unit, g)
            if r is None:
                L.log("06", "skipped", f"{unit} {g}: fewer than two days, no day reference")
                continue
            rows.append(r)
            print(f"  (d) {r['comparison']}: mean best |cos| {r['mean_best_abs_cos']:.3f}  "
                  f"JS {r['js_a_to_b']:.4f} / {r['js_b_to_a']:.4f}")
    table = pd.DataFrame(rows)
    table.insert(0, "unit", unit)
    L.save_csv(table, "06", "consistency", unit, sp, "KvsP")


missing = [g for g in ("K", "P") if g not in RUN_SETS]
if missing:
    msg = f"Step 6 needs K and P in RUN_SETS (missing {missing})"
    print(msg)
    L.log("06", "skipped", msg)
else:
    for unit in RUN_UNITS:
        run_step6(unit)
    L.log("06", "done")


===== FM1 (F->M1)
  (a) best P match of each K PC:
 K_pc  best_P_pc  abs_cos
    1          2    0.767
    2          1    0.766
    3          3    0.737
    4          4    0.820
    5          5    0.786
    6          6    0.825


  fig figures/06_loading_cos__FM1__base__KvsP.png


  fig figures/06_variance_compare__FM1__base__KvsP.png


  fig figures/06_acf_compare__FM1__base__KvsP.png
  (b) 1/e times of matched PCs (s):
 K_pc  P_pc  abs_cos  tau_1e_K_s  tau_1e_P_s
    1     2    0.767         NaN       4.322
    2     1    0.766       4.996         NaN
    3     3    0.737       2.065       1.725
    4     4    0.820       1.834       1.517
    5     5    0.786       1.199       2.296
    6     6    0.825       1.003       1.372


  (c) P in the K state space: outside {'PC1': 3.068567132577443e-06, 'PC2': 3.835708915721804e-05, 'PC3': 0.0002638967734016601}  JS distance 0.5288


  fig figures/06_occupancy_Kspace__FM1__bins10-pcs1_2_3__KvsP.png
      tau 7: diagonal share K 0.7219  P-projected 0.7847  states in both 722
      tau 17: diagonal share K 0.5364  P-projected 0.6245  states in both 721


... (34 outputs left out)

  fig figures/06_occupancy_Pspace__triple__bins10-pcs1_2_3__KvsP.png
      tau 7: diagonal share P 0.9231  K-projected 0.8970  states in both 537
      tau 17: diagonal share P 0.8295  K-projected 0.7735  states in both 532


  fig figures/06_diag_compare_Pspace__triple__bins10-pcs1_2_3__KvsP.png


  (d) K day [1] vs day [2] (same fish, not independent): mean best |cos| 0.735  JS 0.4038 / 0.3844


  (d) P day [1] vs day [2] (same fish, not independent): mean best |cos| 0.694  JS 0.5293 / 0.5169
  csv data/06_consistency__triple__bins10-pcs1_2_3__KvsP.csv  rows 3


## Step 7 — 用手动标记验证簇（只有填好标记设置时才运行；本步骤是唯一读取标记的地方）

**标记数据（由你填，在参数单元里）**：
- 文件格式与路径：`LABEL_SOURCE`（`"boris"` 用仓库里的 BORIS 读取器，`"events_csv"`，`"frames_csv"`）、`LABEL_PATH`
- 粒度：`LABEL_GRANULARITY`（每帧一个标签 / 事件起止）
- 标签对象：`LABEL_TARGET`（每条鱼 / 配对 / 整个场景；`"scene"` 不筛选对象）
- 类别：`LABEL_CLASSES`（求偶、攻击 → 行为名或 BORIS 类别的列表；其余为 other）
- `LABEL_UNLABELED = "other"` 或 `"ignore"`（没标记的帧的处理，会改变所有指标）
- `LABEL_FPS`：标记帧率与追踪帧率不同时填标记帧率，帧号按 追踪fps/标记fps 换算
- `LABEL_PAIR_MAP`：FM1 → M1 的标签，FM2 → M2 的标签，M1M2 → 两条雄鱼之间的标签（BORIS 的 `who` 写法，如 `"M1->F"`、`"M1<->M2"`）

`LABEL_PAIR_MAP` 或 `LABEL_CLASSES` 为空时本步骤跳过，并在 Step 8 的清单里写明原因。

**做什么**：按视频名、帧号对齐标签和 Step 5b 的簇；按组划分（`DEV_SETS → TEST_SETS`，并自动做反方向）；在开发组上学簇→标签映射（纯度 < `MIN_PURITY` 的簇为 unassigned），在测试组上固定使用。输出混淆矩阵（计数和按行归一化）、簇纯度表、预测标签的混淆矩阵和每类召回/精确率/F1、总准确率（旁注多数类基线）、ARI 与 NMI、事件级评估（容差用秒）、时间轴对照图、按视频的 bootstrap 区间、整体平移的 ARI 参照。全部写到 `validation/`。

In [11]:
# Step 7: validation with manual labels. The only cell that reads labels; skipped until configured.
L.clear_log("07")


def step7_problems():
    why = []
    if not LABEL_PAIR_MAP:
        why.append("LABEL_PAIR_MAP is empty")
    if not LABEL_CLASSES:
        why.append("LABEL_CLASSES is empty")
    if LABEL_PAIR_MAP and not any(u in LABEL_PAIR_MAP for u in RUN_UNITS):
        why.append("no unit of RUN_UNITS is in LABEL_PAIR_MAP")
    for s in list(DEV_SETS) + list(TEST_SETS):
        if s not in ("K", "P") or s not in RUN_SETS:
            why.append(f"set {s} must be K or P and be in RUN_SETS")
    if VALIDATION_CLUSTERS == "KP" and "KP" not in RUN_SETS:
        why.append("VALIDATION_CLUSTERS='KP' needs KP in RUN_SETS")
    return why


def class_of(name, category=""):
    for cls, names in LABEL_CLASSES.items():
        lower = {str(x).casefold() for x in names}
        if str(name).casefold() in lower or str(category).casefold() in lower:
            return cls
    return "other"


def label_events(experiment, track_fps):
    # Events of one video (target, cls, start, end inclusive) on the tracking frame clock.
    scale = 1.0 if not LABEL_FPS else track_fps / float(LABEL_FPS)
    if LABEL_SOURCE == "boris":
        from scripts.analysis.behavior_dataset import locate_boris
        from scripts.review.behaviors import default_search, load_bouts
        path, _how = locate_boris(experiment, [Path(LABEL_PATH)] if LABEL_PATH else default_search())
        b = load_bouts(path)
        ev = pd.DataFrame({"target": b["who"].astype(str),
                           "cls": [class_of(x, c) for x, c in zip(b["behavior"], b["category"])],
                           "start": b["start"].astype(float) * scale, "end": b["end"].astype(float) * scale})
    elif LABEL_SOURCE == "events_csv":
        t = pd.read_csv(LABEL_PATH)
        t = t[t["video"] == experiment]
        ev = pd.DataFrame({"target": t["target"].astype(str), "cls": [class_of(x) for x in t["label"]],
                           "start": t["start_s"] * track_fps, "end": t["end_s"] * track_fps})
    elif LABEL_SOURCE == "frames_csv":
        t = pd.read_csv(LABEL_PATH)
        t = t[t["video"] == experiment]
        ev = pd.DataFrame({"target": t["target"].astype(str), "cls": [class_of(x) for x in t["label"]],
                           "start": t["frame"] * scale, "end": t["frame"] * scale})
    else:
        raise ValueError(f"unknown LABEL_SOURCE {LABEL_SOURCE}")
    ev["start"] = np.rint(ev["start"]).astype(int)
    ev["end"] = np.rint(ev["end"]).astype(int)
    return ev


def frame_labels(ev, frames, unit):
    # One label per frame: a class, "other", "" (excluded) ; conflicts are excluded and counted.
    if LABEL_TARGET != "scene":
        want = {str(x).casefold() for x in LABEL_PAIR_MAP.get(unit, [])}
        ev = ev[ev["target"].str.casefold().isin(want)]
    n, f0 = len(frames), int(frames[0])
    lab = np.full(n, "", dtype=object)
    hit = {}
    for r in ev.itertuples():
        a, b = max(r.start - f0, 0), min(r.end - f0 + 1, n)
        if b > a:
            hit.setdefault(r.cls, np.zeros(n, dtype=bool))[a:b] = True
    if "other" in hit:
        lab[hit["other"]] = "other"
    conflict = np.zeros(n, dtype=bool)
    real = np.zeros(n, dtype=bool)
    for cls, m in hit.items():
        if cls == "other":
            continue
        conflict |= real & m
        lab[m] = cls
        real |= m
    lab[conflict] = ""
    if LABEL_UNLABELED == "other":
        lab[(lab == "") & ~conflict] = "other"
    elif LABEL_UNLABELED != "ignore":
        raise ValueError("LABEL_UNLABELED must be 'other' or 'ignore'")
    return lab, int(conflict.sum())


def cluster_frames(unit, tau, dev, target):
    tp = tau_params(unit, tau)
    if VALIDATION_CLUSTERS == "KP":
        lab = L.load_csv("05b", "cluster_labels", unit, tp, "KP")
        return lab[lab["video"].isin(set(load_set(target)["names"]))]
    if target == dev:
        return L.load_csv("05b", "cluster_labels", unit, tp, dev)
    data, seg = load_set(target), load_seg(unit, target)
    info = L.load_json("04", "bin_edges", unit, state_params(unit), dev)
    st, outside = states_in_space(unit, data, seg, load_model(unit, dev), info)
    table = L.load_csv("05b", "state_clusters", unit, tp, dev)["cluster"].to_numpy()
    use = seg >= 0
    cl = np.where(st >= 0, table[np.clip(st, 0, None)], -1)
    print(f"    {target} projected into the {dev} state space; outside the {dev} bins: {outside}")
    return pd.DataFrame({"video": data["names"][data["video"]][use], "frame": data["frame"][use],
                         "state": st[use], "cluster": cl[use]})


def eval_table(unit, tau, dev, target):
    data = load_set(target)
    fps = set_fps(data)
    parts, conflicts = [], 0
    for vi, exp in enumerate(data["names"]):
        frames = data["frame"][data["video"] == vi]
        lab, nc = frame_labels(label_events(exp, fps), frames, unit)
        conflicts += nc
        parts.append(pd.DataFrame({"video": exp, "frame": frames, "manual": lab}))
    manual = pd.concat(parts, ignore_index=True)
    cl = cluster_frames(unit, tau, dev, target)
    merged = manual.merge(cl[["video", "frame", "cluster"]], on=["video", "frame"], how="left")
    merged["cluster"] = merged["cluster"].fillna(-1).astype(int)
    labelled = merged["manual"] != ""
    matched = labelled & (merged["cluster"] >= 0)
    print(f"    {target}: frames {len(merged)}  labelled {int(labelled.sum())}  per class "
          f"{merged.loc[labelled, 'manual'].value_counts().to_dict()}  with a cluster {int(matched.sum())}  "
          f"without {int((labelled & ~matched).sum())} ({100 * (labelled & ~matched).sum() / max(labelled.sum(), 1):.2f}%)  "
          f"conflicting frames dropped {conflicts}")
    return merged, fps


def fit_mapping(ev):
    rows = []
    for c, part in ev.groupby("cluster"):
        vc = part["manual"].value_counts()
        purity = float(vc.iloc[0] / vc.sum())
        rows.append({"cluster": int(c), "n_frames": int(vc.sum()), "top_label": vc.index[0], "purity": purity,
                     "mapped_label": vc.index[0] if purity >= MIN_PURITY else "unassigned"})
    return pd.DataFrame(rows)


def runs_of(labels, frames, keep):
    if len(labels) == 0:
        return []
    brk = np.ones(len(labels), dtype=bool)
    brk[1:] = (labels[1:] != labels[:-1]) | (np.diff(frames) != 1)
    starts = np.flatnonzero(brk)
    ends = np.r_[starts[1:], len(labels)]
    return [(labels[a], int(frames[a]), int(frames[b - 1])) for a, b in zip(starts, ends) if labels[a] in keep]


def match_events(manual_ev, pred_ev, fps):
    hit_e = np.zeros(len(manual_ev), dtype=bool)
    ok_p = np.zeros(len(pred_ev), dtype=bool)
    by_cls = {}
    for j, (c, s, e) in enumerate(pred_ev):
        by_cls.setdefault(c, []).append((s, e, j))
    arrays = {c: (np.array([x[0] for x in v]), np.array([x[1] for x in v]), np.array([x[2] for x in v]))
              for c, v in by_cls.items()}
    for i, (c, s, e) in enumerate(manual_ev):
        if c not in arrays:
            continue
        tol = int(round(min(TOL_S, TOL_FRAC * (e - s + 1) / fps) * fps))
        ps, pe, pj = arrays[c]
        lo = np.searchsorted(pe, s - tol, side="left")
        hi = np.searchsorted(ps, e + tol, side="right")
        for k in range(lo, hi):
            a, b = ps[k], pe[k]
            if abs(a - s) <= tol:
                a = s
            if abs(b - e) <= tol:
                b = e
            if max(0, min(b, e) - max(a, s) + 1) / (e - s + 1) >= OVERLAP_THR:
                hit_e[i] = True
                ok_p[pj[k]] = True
    return hit_e, ok_p


def duration_bin(frames_len, fps):
    d = frames_len / fps
    edges = [0.0] + list(DURATION_EDGES_S) + [np.inf]
    for a, b in zip(edges[:-1], edges[1:]):
        if a <= d < b:
            return f"{a:g}-{b:g} s" if np.isfinite(b) else f">={a:g} s"
    return "?"


def confusion_fig(mat, rows, cols, title, normalize):
    show = mat.astype(float)
    if normalize:
        rs = show.sum(axis=1, keepdims=True)
        show = np.divide(show, rs, out=np.zeros_like(show), where=rs > 0)
    fig, ax = plt.subplots(figsize=(min(30, max(6, 0.3 * len(cols) + 3)), max(3, 0.6 * len(rows) + 2)))
    im = ax.imshow(show, aspect="auto", cmap="viridis", vmin=0, vmax=1 if normalize else None, interpolation="nearest")
    ax.set_xticks(np.arange(len(cols)), cols, rotation=90, fontsize=6)
    ax.set_yticks(np.arange(len(rows)), rows, fontsize=8)
    ax.set_ylabel("manual label")
    fig.colorbar(im, ax=ax, fraction=0.03, label="row fraction" if normalize else "frames")
    ax.set_title(title, fontsize=9)
    return fig


def evaluate(unit, tau, dev, test):
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, precision_recall_fscore_support

    direction = f"{dev}-to-{test}"
    vp = {**tau_params(unit, tau), "clu": VALIDATION_CLUSTERS}
    print(f"\n[{unit_title(unit)} tau {tau}] {direction}")
    tables = {}
    for role, target in (("dev", dev), ("test", test)):
        merged, fps = eval_table(unit, tau, dev, target)
        tables[role] = (merged[(merged["manual"] != "") & (merged["cluster"] >= 0)].copy(), merged, fps)
    mapping = fit_mapping(tables["dev"][0])
    L.save_csv(mapping, "07", "cluster_label_map", unit, vp, direction, folder="validation", verbose=False)
    mp = dict(zip(mapping["cluster"], mapping["mapped_label"]))
    classes = list(LABEL_CLASSES)
    out_rows = []
    for role in ("dev", "test"):
        ev, merged, fps = tables[role]
        if ev.empty:
            print(f"    {role}: no labelled frame with a cluster")
            continue
        ev["pred"] = [mp.get(c, "unassigned") for c in ev["cluster"]]
        clusters = sorted(ev["cluster"].unique(), key=lambda c: (mp.get(c, "unassigned"), c))
        man_labels = sorted(ev["manual"].unique())
        ct = pd.crosstab(ev["manual"], ev["cluster"]).reindex(index=man_labels, columns=clusters, fill_value=0)
        cols = [f"C{c}:{mp.get(c, 'unassigned')}" for c in clusters]
        for norm, nm in ((False, "counts"), (True, "rownorm")):
            fig = confusion_fig(ct.to_numpy(), man_labels, cols, f"{unit_title(unit)} tau {tau} {direction} {role}: "
                                f"manual x cluster ({nm}); columns grouped by mapped label", norm)
            save_fig(fig, "07", f"confusion_cluster_{nm}_{role}", unit, vp, direction, folder="validation")
        pur = []
        for c in clusters:
            vc = ev.loc[ev["cluster"] == c, "manual"].value_counts()
            pur.append({"cluster": c, "n_frames": int(vc.sum()), "top_label": vc.index[0],
                        "purity": float(vc.iloc[0] / vc.sum()), "mapped_label_from_dev": mp.get(c, "unassigned")})
        L.save_csv(pd.DataFrame(pur), "07", f"cluster_purity_{role}", unit, vp, direction, folder="validation", verbose=False)
        pred_labels = sorted(set(ev["pred"]) | set(man_labels))
        cp = pd.crosstab(ev["manual"], ev["pred"]).reindex(index=man_labels, columns=pred_labels, fill_value=0)
        for norm, nm in ((False, "counts"), (True, "rownorm")):
            fig = confusion_fig(cp.to_numpy(), man_labels, pred_labels, f"{unit_title(unit)} tau {tau} {direction} "
                                f"{role}: manual x predicted ({nm})", norm)
            save_fig(fig, "07", f"confusion_pred_{nm}_{role}", unit, vp, direction, folder="validation")
        labs = sorted(set(man_labels))
        p, r, f, sup = precision_recall_fscore_support(ev["manual"], ev["pred"], labels=labs, zero_division=0)
        acc = float((ev["manual"] == ev["pred"]).mean())
        majority = float(ev["manual"].value_counts(normalize=True).iloc[0])
        sc = pd.DataFrame({"label": labs, "recall": r, "precision": p, "f1": f, "support": sup})
        sc.loc[len(sc)] = ["accuracy (majority-class baseline in 'precision')", acc, majority, np.nan, len(ev)]
        L.save_csv(sc, "07", f"scores_{role}", unit, vp, direction, folder="validation", verbose=False)
        cr = ev["manual"].isin(classes)
        ari = {"all_labelled": (adjusted_rand_score(ev["manual"], ev["cluster"]),
                                normalized_mutual_info_score(ev["manual"], ev["cluster"])),
               "courtship_attack_only": (adjusted_rand_score(ev.loc[cr, "manual"], ev.loc[cr, "cluster"]) if cr.sum() else np.nan,
                                         normalized_mutual_info_score(ev.loc[cr, "manual"], ev.loc[cr, "cluster"]) if cr.sum() else np.nan)}
        L.save_csv(pd.DataFrame([{"frames": k, "ARI": v[0], "NMI": v[1]} for k, v in ari.items()]),
                   "07", f"ari_nmi_{role}", unit, vp, direction, folder="validation", verbose=False)
        print(f"    {role}: frames {len(ev)}  accuracy {acc:.3f} (majority baseline {majority:.3f})  "
              f"ARI/NMI all {ari['all_labelled'][0]:.3f}/{ari['all_labelled'][1]:.3f}")
        print("    " + sc.round(3).to_string(index=False).replace("\n", "\n    "))
        # (e) event-level evaluation
        ev_rows = []
        for vid, part in merged.groupby("video", sort=False):
            m_frames = part.loc[part["manual"] != "", "frame"].to_numpy()
            m_lab = part.loc[part["manual"] != "", "manual"].to_numpy()
            pp = ev[ev["video"] == vid]
            m_ev = runs_of(m_lab, m_frames, set(classes))
            p_ev = runs_of(pp["pred"].to_numpy(), pp["frame"].to_numpy(), set(classes))
            hit, okp = match_events(m_ev, p_ev, fps)
            for (c, s, e), h in zip(m_ev, hit):
                ev_rows.append({"kind": "manual", "video": vid, "cls": c, "dur": duration_bin(e - s + 1, fps), "ok": h})
            for (c, s, e), o in zip(p_ev, okp):
                ev_rows.append({"kind": "pred", "video": vid, "cls": c, "dur": duration_bin(e - s + 1, fps), "ok": o})
        er = pd.DataFrame(ev_rows, columns=["kind", "video", "cls", "dur", "ok"])
        res = []
        for (cls, dur), g in er.groupby(["cls", "dur"]):
            mm, pq = g[g["kind"] == "manual"], g[g["kind"] == "pred"]
            res.append({"label": cls, "duration": dur, "n_manual_events": len(mm),
                        "event_recall": mm["ok"].mean() if len(mm) else np.nan,
                        "n_pred_events": len(pq), "event_precision": pq["ok"].mean() if len(pq) else np.nan})
        for cls, g in er.groupby("cls"):
            mm, pq = g[g["kind"] == "manual"], g[g["kind"] == "pred"]
            res.append({"label": cls, "duration": "all", "n_manual_events": len(mm),
                        "event_recall": mm["ok"].mean() if len(mm) else np.nan,
                        "n_pred_events": len(pq), "event_precision": pq["ok"].mean() if len(pq) else np.nan})
        L.save_csv(pd.DataFrame(res), "07", f"events_{role}", unit, vp, direction, folder="validation", verbose=False)
        if role != "test":
            continue
        # (f) timelines of example test videos
        colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(classes)}
        for vid in list(merged["video"].unique())[:N_EXAMPLE]:
            part = merged[merged["video"] == vid]
            f0 = int(part["frame"].iloc[0])
            pp = ev[ev["video"] == vid]
            fig, ax = plt.subplots(figsize=(16, 2.4))
            for y, labs_, frs in ((1, part["manual"].to_numpy(), part["frame"].to_numpy()),
                                  (0, pp["pred"].to_numpy(), pp["frame"].to_numpy())):
                for c, s, e in runs_of(labs_, frs, set(classes)):
                    ax.broken_barh([((s - f0) / fps, (e - s + 1) / fps)], (y - 0.4, 0.8), color=colors[c])
            ax.set_yticks([0, 1], ["predicted", "manual"])
            ax.set_xlabel("time (s)")
            ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=v, label=k) for k, v in colors.items()], fontsize=8,
                      loc="upper right")
            ax.set_title(f"{vid} {unit_title(unit)} tau {tau}: manual vs predicted ({direction})", fontsize=9)
            save_fig(fig, "07", f"timeline_{vid}", unit, vp, direction, folder="validation")
        # 5. robustness: bootstrap over videos and per-video shifts
        rng = np.random.default_rng(VALIDATION_SEED)
        vids = ev["video"].unique()
        per = {v: ev[ev["video"] == v] for v in vids}
        boot = []
        for _ in range(BOOT):
            pick = rng.choice(vids, size=len(vids), replace=True)
            sub = pd.concat([per[v] for v in pick])
            p, r, f, _s = precision_recall_fscore_support(sub["manual"], sub["pred"], labels=classes, zero_division=0)
            boot.append(np.r_[r, p])
        boot = np.array(boot)
        bt = pd.DataFrame({"label": classes,
                           "recall_lo": np.percentile(boot[:, :len(classes)], 2.5, axis=0),
                           "recall_hi": np.percentile(boot[:, :len(classes)], 97.5, axis=0),
                           "precision_lo": np.percentile(boot[:, len(classes):], 2.5, axis=0),
                           "precision_hi": np.percentile(boot[:, len(classes):], 97.5, axis=0),
                           "n_videos": len(vids), "note": "videos are not independent samples"})
        L.save_csv(bt, "07", "bootstrap_test", unit, vp, direction, folder="validation", verbose=False)
        shifts = []
        for _ in range(SHIFT_N):
            sh = []
            for v in vids:
                c = per[v]["cluster"].to_numpy()
                sh.append(np.roll(c, int(rng.integers(1, max(len(c), 2)))))
            shifts.append(adjusted_rand_score(ev["manual"], np.concatenate(sh)))
        out_rows.append({"unit": unit, "tau": tau, "direction": direction, "test_accuracy": acc,
                         "majority_baseline": majority, "test_ARI": ari["all_labelled"][0],
                         "shift_ARI_min": np.min(shifts), "shift_ARI_mean": np.mean(shifts), "shift_ARI_max": np.max(shifts)})
        print(f"    shifted-label ARI x{SHIFT_N}: [{np.min(shifts):.4f}, {np.max(shifts):.4f}] vs observed {ari['all_labelled'][0]:.4f}")
    if out_rows:
        L.save_csv(pd.DataFrame(out_rows), "07", "summary", unit, vp, direction, folder="validation", verbose=False)


problems = step7_problems()
if problems:
    msg = "Step 7 not run: " + "; ".join(problems)
    print(msg)
    L.log("07", "skipped", msg)
else:
    for unit in [u for u in RUN_UNITS if u in LABEL_PAIR_MAP]:
        for tau in TAUS:
            for dev, test in [(d, t) for d in DEV_SETS for t in TEST_SETS] + [(t, d) for d in DEV_SETS for t in TEST_SETS]:
                evaluate(unit, tau, dev, test)
    L.log("07", "done")

Step 7 not run: LABEL_PAIR_MAP is empty; LABEL_CLASSES is empty


## Step 8 — 总览与对比（最后一个单元；运行完 setup 单元后也可以单独重复运行）

1. 对比图（`compare/`，把 `figures/` 里已有的图并排拼接，加标题，缺图跳过，宽度不超过约 4000 px）：`by_set`（K | P | KP）、`by_tau`（Step 5、5b 的 τ=7 | τ=17）、`by_unit`（FM1 | FM2 | M1M2 | triple）、`by_param-<key>`（某个参数有多个取值时）。文件名 `cmp_<类型>__<步骤>__<固定的其他字段>.png`。
2. `index.html`：单个文件，不依赖外部库；顶部三个下拉筛选（步骤、分析集、单元），按步骤分节显示缩略图，点击在页内放大；另一个标签页是全部对比图；顶部是 `params.json` 的摘要。全是相对路径，整个文件夹拷走也能打开。
3. `summary.csv`：每个（分析集, 单元, τ, 状态模式）一行，外加 Step 6 的比较行。
4. 辅助函数 `show(step, unit, set_name, name=None, **params)` 和 `compare(step, vary="set"|"tau"|"unit", name=None, **fixed)`。
5. 最终清单：图的数量、被跳过的步骤及原因。

In [12]:
# Step 8: comparison images, summary.csv, index.html, display helpers, final checklist.
from IPython.display import Image as _IPImage, display

t0 = time.time()
CMP = S.build_compares(L)
print(f"comparison images: {len(CMP)}  ({time.time() - t0:.0f} s)")


def _try_csv(*args, **kw):
    try:
        return L.load_csv(*args, **kw)
    except FileNotFoundError:
        return None


def _try_json(*args):
    try:
        return L.load_json(*args)
    except FileNotFoundError:
        return None


rows = []
for set_name in RUN_SETS:
    for unit in RUN_UNITS:
        p1, sp = unit_params(unit), state_params(unit)
        try:
            seg = load_seg(unit, set_name)
        except FileNotFoundError:
            continue
        base = {"set": set_name, "unit": unit, "valid_frames": int((seg >= 0).sum()),
                "missing_rate": float((seg < 0).mean())}
        ev = _try_csv("01", "explained_variance", unit, p1, set_name)
        if ev is not None:
            base["n_pc_90pct"] = int(np.searchsorted(ev["cumulative_percent"].to_numpy(), 90.0) + 1)
        at = _try_csv("03", "acf_times", unit, p1, set_name)
        if at is not None:
            for i in range(min(3, len(at))):
                base[f"acf_1e_pc{i + 1}_s"] = at.loc[i, "tau_1e_s"]
        info = _try_json("04", "bin_edges", unit, sp, set_name)
        for tau in TAUS:
            tp = tau_params(unit, tau)
            if DO_JOINT:
                r = {**base, "tau": tau, "mode": "joint", "params": S.param_label(tp)}
                if info:
                    r["occupied_states"] = info.get("occupied_joint")
                    r["grid_states"] = info["n_states"]
                ws = _try_csv("05b", "within_summary", unit, tp, set_name)
                if ws is not None:
                    w = ws.iloc[0]
                    r.update({"n_clusters": int(w["n_clusters"]),
                              "largest_cluster_state_fraction": w["largest_cluster_state_fraction"],
                              "largest_cluster_frame_fraction": w["largest_cluster_frame_fraction"],
                              "within_fraction": w["within_fraction"], "within_fraction_nodiag": w["within_fraction_nodiag"],
                              "shuffle_range": f"{w['shuffle_min']:.4f}-{w['shuffle_max']:.4f}",
                              "shuffle_nodiag_range": f"{w['shuffle_nodiag_min']:.4f}-{w['shuffle_nodiag_max']:.4f}"})
                rows.append(r)
            if DO_PER_PC:
                r = {**base, "tau": tau, "mode": "per_pc", "params": S.param_label(tp)}
                if info:
                    r["occupied_states"] = "/".join(str(v) for v in info.get("occupied_per_var", []))
                    r["grid_states"] = "/".join(str(v) for v in info["widths"])
                rows.append(r)
for unit in RUN_UNITS:
    cs = _try_csv("06", "consistency", unit, state_params(unit), "KvsP")
    if cs is not None:
        for r in cs.to_dict("records"):
            rows.append({"set": r.pop("comparison"), "unit": unit, "mode": "consistency",
                         "params": S.param_label(state_params(unit)), **{k: v for k, v in r.items() if k != "unit"}})
summary = pd.DataFrame(rows)
summary.to_csv(L.root / "summary.csv", index=False)
print(f"summary.csv: {len(summary)} rows")

params_all = json.loads(L.params_json.read_text(encoding="utf-8"))
idx = L.read_index()
lines = [f"run {RUN_NAME}; written {time.strftime('%Y-%m-%d %H:%M')}; sets {RUN_SETS}; units {RUN_UNITS}",
         f"videos: {', '.join(params_all.get('frame_counts', {}).get('videos', {}).keys())}",
         f"preprocessing {S.param_label(pre_params())}; STATE_VARS {STATE_VARS}; TAUS {TAUS}",
         f"Infomap: {params_all.get('infomap', {}).get('backend')} {params_all.get('infomap', {}).get('version')}, "
         f"trials {N_TRIALS} (scan {N_TRIALS_SCAN}), seed {INFOMAP_SEED}",
         f"{len(idx)} figures, {len(list(L.compare.glob('cmp_*.png')))} comparison images"]
html_path = S.build_index_html(L, f"Pose PCA / microstates / Infomap - {RUN_NAME}", lines)
print("index:", html_path)


def _step_code(step):
    return f"{int(step):02d}" if str(step).isdigit() else str(step)


def _match_params(label, params):
    parsed = S.parse_label(label)
    for k, v in params.items():
        want = "" if v is True else S._fmt_value(v)
        if parsed.get(k) != want:
            return False
    return True


def show(step, unit, set_name, name=None, **params):
    # Display the saved figures of one step / unit / set (optionally one name and parameter values).
    idx = L.read_index()
    sel = idx[(idx["step"] == _step_code(step)) & (idx["unit"] == unit) & (idx["set"] == set_name)]
    if name:
        sel = sel[sel["name"] == name]
    if params:
        sel = sel[[_match_params(p, params) for p in sel["params"]]]
    for r in sel.itertuples():
        print(r.file)
        display(_IPImage(filename=str(L.root / r.folder / r.file)))
    return sel[["file", "title"]]


def compare(step, vary="set", name=None, **fixed):
    # Side-by-side view across sets, taus or units, built on the fly from figures/.
    idx = L.read_index()
    idx = idx[(idx["folder"] == "figures") & (idx["step"] == _step_code(step))].copy()
    if name:
        idx = idx[idx["name"] == name]
    unit = fixed.pop("unit", None)
    set_name = fixed.pop("set_name", None)
    if unit and vary != "unit":
        idx = idx[idx["unit"] == unit]
    if set_name and vary != "set":
        idx = idx[idx["set"] == set_name]
    if vary == "tau":
        fixed.pop("tau", None)
    if fixed:
        idx = idx[[_match_params(p, fixed) for p in idx["params"]]]
    if vary == "set":
        idx["v"], idx["k"], order = idx["set"], idx["name"] + "|" + idx["unit"] + "|" + idx["params"], S.SET_ORDER
    elif vary == "unit":
        idx["v"] = idx["unit"]
        idx["k"] = idx["name"] + "|" + idx["set"] + "|" + [S.join_label({a: b for a, b in S.parse_label(p).items()
                                                                          if a != "corot"}) for p in idx["params"]]
        order = S.UNIT_ORDER
    elif vary == "tau":
        idx["v"] = [S.parse_label(p).get("tau", "") for p in idx["params"]]
        idx["k"] = idx["name"] + "|" + idx["unit"] + "|" + idx["set"] + "|" + [
            S.join_label({a: b for a, b in S.parse_label(p).items() if a != "tau"}) for p in idx["params"]]
        order = tuple(str(t) for t in sorted({int(v) for v in idx["v"] if v}))
    else:
        raise ValueError("vary must be 'set', 'tau' or 'unit'")
    shown = 0
    for key, g in idx.groupby("k"):
        g = g.sort_values("v", key=lambda s: s.map(lambda x: order.index(x) if x in order else 99))
        if len(g) < 2:
            continue
        out = L.compare / f"adhoc_{vary}__{_step_code(step)}__{key.replace('|', '__')}.png"
        p = S.stack_images([L.figures / f for f in g["file"]], [f"{vary}={v}" for v in g["v"]], key.replace("|", "  "), out)
        if p:
            display(_IPImage(filename=str(p)))
            shown += 1
    print(f"{shown} comparison image(s)")


# Final checklist
idx = L.read_index()
print("\n===== checklist")
print(f"figures: {int((idx['folder'] == 'figures').sum())} in figures/, {int((idx['folder'] == 'validation').sum())} in validation/")
print(f"comparison images: {len(list(L.compare.glob('cmp_*.png')))}; data files: {len(list(L.data.iterdir()))}")
print("figures per step:", idx.groupby("step").size().to_dict())
log = L.read_log()
done = sorted({e["step"] for e in log if e["status"] == "done"})
print("steps done:", done)
skipped = [e for e in log if e["status"] == "skipped"]
if skipped:
    print("skipped:")
    for e in skipped:
        print(f"  step {e['step']}: {e['reason']}")
else:
    print("skipped: none")
print("output folder:", L.root)

comparison images: 441  (228 s)


summary.csv: 60 rows


index: \\bucket.oist.jp\bucket\StephensU\medaka\runs\behavior_features\outputs\20261107_pose_states\index.html

===== checklist
figures: 485 in figures/, 0 in validation/
comparison images: 441; data files: 330
figures per step: {'00': 13, '01': 24, '02': 36, '03': 12, '04': 24, '05': 48, '05b': 300, '06': 28}
steps done: ['00', '01', '02', '03', '04', '05', '05b', '06']
skipped:
  step 05b: interactive plotly heatmaps (MAKE_HTML = False)
  step 07: Step 7 not run: LABEL_PAIR_MAP is empty; LABEL_CLASSES is empty
output folder: \\bucket.oist.jp\bucket\StephensU\medaka\runs\behavior_features\outputs\20261107_pose_states
